# Phase 0 — LiDAR Free-Space Map

**Goal (§28 Phase 0, items 0.1–0.2):** Build a global per-frame occupancy/free-space voxel map
by ray-casting the raw LiDAR sweep, then visualise it.

**Method (§5, §6):** For each LiDAR beam, use Amanatides & Woo fast voxel traversal to:
- Mark every voxel *between* sensor and return point as **FREE** (negative log-odds)
- Mark the *return* voxel as **OCCUPIED** (positive log-odds)
- Leave all other voxels as **UNKNOWN** (zero log-odds = uninitialised prior)

The map is built once per frame, globally, **object-agnostic and mask-agnostic** (§6).  
It is then queried per GT box to compute free/occupied/unknown volume fractions.

**Key rules (§29):**
- Do NOT ground-filter the cloud used for ray casting — ground returns are excellent free-space evidence
- Use single sweep only — no aggregation (§25.2)
- Free-space evidence comes only from real LiDAR beams, never from depth predictions (§9)

**Datasets:** ECP (64-beam Velodyne) and nuScenes mini (32-beam Velodyne).  
Compare them side-by-side — the 32-beam density question is an open question (§30 item 1).

## 0. Configuration

Change `DATASET` to switch between ECP and nuScenes mini.  
Provide specific scene/frame when you know them; defaults are placeholders.

> **Fixed 2026-08-31 — three compounding bugs in the free-space map, found while
> checking why nuScenes BEV renders looked "off" (free space stopping short of
> occupied returns instead of reaching them). Two affect the actual reported
> statistics; one is display-only.**
>
> 1. **`THRESHOLD_FREE` miscalibrated for single-sweep mode (affects real numbers).**
>    A single ray traversal only contributes one `LO_FREE = -0.4` step, but the old
>    threshold (`-0.5`) required the equivalent of *two* independent ray crossings
>    through the same voxel to certify FREE. Since this pipeline deliberately uses a
>    single sweep with no temporal aggregation (§25.2), most voxels are only ever
>    crossed once — so ~73% of genuinely swept-through free space was silently
>    reclassified as UNKNOWN (verified: 72.8% of touched voxels in a real nuScenes
>    frame sat at exactly one `LO_FREE` step and were being thrown away). This fed
>    into every OBB query (`query_box_freespace_obb` uses the real 3D grid directly),
>    so it affects every `frac_free`-derived number in `phase0_results.md`.
>    `THRESHOLD_FREE = -0.2` now certifies FREE off a single ray pass, which is the
>    correct threshold for a design with no repeated observations to accumulate.
>    Also worse for nuScenes' sparser 32-beam array and worse again at finer voxel
>    sizes (rarer double-hits either way) — a second, previously undocumented
>    mechanism behind the §3 resolution-sensitivity shrinkage, on top of that
>    section's stated cause.
>
> 2. **BEV/cross-section Z-band collapse used a raw `.max()` on signed log-odds
>    (display-only — does NOT affect any reported statistic).** Untouched voxels are
>    exactly `0`, free voxels are negative, and `0 > any negative value` — so a single
>    untouched voxel anywhere in the visualized height band masked real free-space
>    evidence from every other layer in that column. This was the dominant reason the
>    pictures looked wrong (free space forming small blobs / rings instead of reaching
>    outward to occupied returns) — confirmed by measurement: an any()-based,
>    occupied-takes-priority reduction gave **280,585** free BEV pixels on a test
>    frame vs. **5,877** from the old `.max()` reduction, everything else held fixed.
>    Fixed in cells 16, 22, and 37. `query_box_freespace_obb` was never affected — it
>    always indexed the real 3D grid directly, never this 2D collapse.
>
> 3. **`BEV_Z_HALF_WIDTH = VOXEL_SIZE * 3` (display-only).** Floored at 0.5 m so the
>    visualized slice stays a representative physical thickness at fine voxel sizes;
>    at 0.05 m the un-floored formula gave a 0.3 m-thick band that diverging LiDAR
>    rings barely crossed beyond a few meters, independent of bug #2 above.
>
> 4. **Demo cells anchored the "ego frame" on the camera's pose instead of the LiDAR's
> (display-only — `experiment_a_batch.py` / `experiment_a_matched_pairs.py` always used
> the LiDAR's own ego pose and were never affected, but every single-frame demo figure
> in this notebook was).** GT boxes, predicted boxes, and every downstream OBB query were
> transformed into the ego frame using `R_e2g`/`t_e2g` from the anchor **camera's** ego
> pose (§2), while the LiDAR sweep itself only gets a rigid sensor→ego transform (§3) —
> no pose correction — so it's implicitly expressed in the ego frame at the **LiDAR's
> own** timestamp instead. Camera and LiDAR keyframes can differ by tens of ms; confirmed
> on this dataset: up to ~28ms, i.e. ~21cm of ego motion — enough to visibly misalign a
> GT box from the LiDAR silhouette it's supposed to bound. Fixed in §3 (overwrites
> `R_e2g`/`t_e2g` with the LiDAR's own ego pose right after it's fetched) — every later
> cell picks up the corrected anchor automatically.
>
> **`phase0_results.md` has since been updated for fix #1 and the full pipeline rerun at
> all three voxel sizes (2026-08-31)** — its numeric claims are current as of that fix.
> Fix #4 above (2026-08-31, later the same day) does not change any of those numbers
> (display-only, as noted), but any demo figure generated *before* fix #4 was applied
> (including earlier same-day reruns) has the ~cm-scale box misalignment and should be
> regenerated. A memory guard (`resource.setrlimit`, top of the next cell) was also added
> after a kernel OOM froze the system while testing this — restart the kernel if you ever
> hit `MemoryError`, don't just re-run the same cell, the kernel's memory usage is
> cumulative across runs.


In [ ]:
# ── Self memory guard (added 2026-08-31) ────────────────────────────────────
# This kernel is long-lived and gets re-run across dataset switches; a leaked
# NuScenes(v1.0-trainval) index (~7.4 GB) plus a fresh 0.05 m grid (~2.4 GB) can
# push total usage past physical RAM into swap and freeze the whole desktop.
# Capping this process's virtual memory makes it fail with a clean MemoryError
# instead. If you hit that error, restart the kernel (this releases everything)
# before re-running -- don't just re-run the same cell, the leak is cumulative.
import resource
_MEM_CAP_GB = 12
resource.setrlimit(resource.RLIMIT_AS, (_MEM_CAP_GB * 1024**3, resource.RLIM_INFINITY))
print(f'Kernel memory capped at {_MEM_CAP_GB} GB (RLIMIT_AS) -- restart kernel to release leaked memory.')

# ══════════════════════════════════════════════════════════════════════════════
# PRIMARY SELECTOR — change this to switch dataset
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'nuscenes_mini'    # 'ecp'  or  'nuscenes_mini'

# ── Per-dataset defaults ────────────────────────────────────────────────────
if DATASET == 'ecp':
    DATA_ROOT    = '/workspace/data/ecp'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10         # GT available for scenes 10, 11, 15
    FRAME_IDX    = 560        # Set to a keyframe so GT is available
    LIDAR_LINES  = 64
    # ECP has 3 front-arc cameras; we use FRONT for the FOV frustum overlay
    CAMERAS      = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT']
    CAMERA       = 'CAM_FRONT'

elif DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/workspace/data/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 3          # mini has 10 scenes (0–9)
    FRAME_IDX    = 20
    LIDAR_LINES  = 32
    CAMERAS      = ['CAM_FRONT', 'CAM_FRONT_LEFT', 'CAM_FRONT_RIGHT',
                     'CAM_BACK', 'CAM_BACK_LEFT', 'CAM_BACK_RIGHT']
    CAMERA       = 'CAM_FRONT'

else:
    raise ValueError(f'Unknown DATASET: {DATASET!r}')

# ── Voxel grid parameters ───────────────────────────────────────────────────
VOXEL_SIZE   = 0.05    # [m] — settled resolution after the 0.2/0.1/0.05m sensitivity check

# Grid extent in ego frame (ego vehicle at origin, +X forward, +Y left, +Z up)
# Chosen to cover the full LiDAR FOV without wasting RAM
GRID_X_MIN, GRID_X_MAX = -50.0, 50.0   # [m] forward/backward
GRID_Y_MIN, GRID_Y_MAX = -50.0, 50.0   # [m] left/right
GRID_Z_MIN, GRID_Z_MAX =  -2.0,  6.0   # [m] vertical

# ── Log-odds parameters (OctoMap defaults) ──────────────────────────────────
LO_FREE     = -0.4     # log-odds decrement for each traversal (free evidence)
LO_OCC      =  0.85    # log-odds increment for return voxel  (occupied evidence)
LO_MIN      = -5.0     # clamping floor (saturates free certainty)
LO_MAX      = 10.0     # clamping ceiling (saturates occupied certainty)

# Thresholds for labeling voxels:
#   log-odds < THRESHOLD_FREE  →  CERTIFIED EMPTY
#   log-odds > THRESHOLD_OCC   →  OCCUPIED
#   in between                 →  UNKNOWN
THRESHOLD_FREE = -0.2    # NOTE: fixed 2026-08-31 (was -0.5) -- see the note below the cell
THRESHOLD_OCC  =  0.5

# ── BEV visualisation ───────────────────────────────────────────────────────
BEV_Z_SLICE_HEIGHT = 1.0    # [m above ground] — BEV slice Z in ego frame
BEV_Z_HALF_WIDTH   = max(VOXEL_SIZE * 3, 0.5)   # display-only slice thickness;
    # floored at 0.5m so the BEV picture stays representative at fine voxel sizes --
    # a bare VOXEL_SIZE*3 band is only 0.3m thick at 0.05m and diverging LiDAR rings
    # barely cross it beyond a few meters, even though the real 3D free-space grid
    # (and every OBB query / reported statistic, which use the box's true height, not
    # this slice) is unaffected. Purely cosmetic; doesn't change any number in the report.
BEV_MAX_RANGE_PLOT  = 40.0  # [m] only show this radius in BEV

# ── Ray casting options ─────────────────────────────────────────────────────
RAY_MIN_RANGE = 0.5    # [m] skip returns closer than this (avoids sensor self-hit)
RAY_MAX_RANGE = 80.0   # [m] skip returns further than this (noise)

# ── Paths ───────────────────────────────────────────────────────────────────
from pathlib import Path
DEV_ROOT = Path('/workspace')

print(f'Dataset  : {DATASET}')
print(f'Scene    : {SCENE_IDX}, Frame: {FRAME_IDX}')
print(f'LiDAR    : {LIDAR_LINES}-beam')
print(f'Voxel    : {VOXEL_SIZE} m')
print(f'Grid     : X[{GRID_X_MIN},{GRID_X_MAX}] Y[{GRID_Y_MIN},{GRID_Y_MAX}] Z[{GRID_Z_MIN},{GRID_Z_MAX}] m')
_nx = int((GRID_X_MAX - GRID_X_MIN) / VOXEL_SIZE)
_ny = int((GRID_Y_MAX - GRID_Y_MIN) / VOXEL_SIZE)
_nz = int((GRID_Z_MAX - GRID_Z_MIN) / VOXEL_SIZE)
print(f'Grid dim : {_nx} × {_ny} × {_nz} = {_nx*_ny*_nz/1e6:.1f} M voxels  '
      f'({_nx*_ny*_nz*4/1024**2:.1f} MB float32)')

## 1. Imports

In [ ]:
import sys
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

# ── Numba for fast JIT-compiled ray casting ──────────────────────────────────
try:
    import numba
    from numba import njit, prange
    HAS_NUMBA = True
    print('numba available — using JIT ray caster')
except ImportError:
    HAS_NUMBA = False
    print('WARNING: numba not found — falling back to numpy ray caster (slower)')

print('Imports OK')

## 2. NuScenes init + frame load

Using the same data-loading pattern as `Testing/autolabeling_pipeline.ipynb`.

In [ ]:
# ── Init NuScenes ────────────────────────────────────────────────────────────
nusc = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)

scene_rec  = nusc.scene[SCENE_IDX]
scene_name = scene_rec['name']
print(f'Scene: {SCENE_IDX} → {scene_name}  ({scene_rec["nbr_samples"]} keyframes)')

# ── Helper: load one camera frame ────────────────────────────────────────────
def load_cam_frame(nusc, scene_idx, frame_idx, camera):
    """Walk the sample_data chain to frame_idx, return intrinsics + extrinsics."""
    scene   = nusc.scene[scene_idx]
    token   = nusc.get('sample', scene['first_sample_token'])['data'][camera]
    for _ in range(frame_idx):
        rec   = nusc.get('sample_data', token)
        token = rec['next']
        assert token, f'frame_idx {frame_idx} exceeds scene length'
    rec  = nusc.get('sample_data', token)
    cal  = nusc.get('calibrated_sensor', rec['calibrated_sensor_token'])
    K    = np.array(cal['camera_intrinsic'], dtype=np.float64)
    R_c2e = Quaternion(cal['rotation']).rotation_matrix.astype(np.float64)
    t_c2e = np.array(cal['translation'], dtype=np.float64)
    import cv2
    img_path = nusc.get_sample_data_path(token)
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    return dict(img=img, K=K, R_c2e=R_c2e, t_c2e=t_c2e,
                H=H, W=W, path=img_path, sd_token=token,
                is_key=rec['is_key_frame'])

# ── Load all cameras ──────────────────────────────────────────────────────────
cam_data = {}
for cam in CAMERAS:
    cam_data[cam] = load_cam_frame(nusc, SCENE_IDX, FRAME_IDX, cam)
    cd = cam_data[cam]
    print(f'  {cam}: {cd["W"]}×{cd["H"]}  is_key={cd["is_key"]}  {cd["path"]}')

# ── Ego pose at anchor timestamp ──────────────────────────────────────────────
anchor_sd  = nusc.get('sample_data', cam_data[CAMERA]['sd_token'])
anchor_ep  = nusc.get('ego_pose', anchor_sd['ego_pose_token'])
R_e2g = Quaternion(anchor_ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g = np.array(anchor_ep['translation'], dtype=np.float64)

print(f'\nPrimary camera: {CAMERA}')
print(f'Frame {FRAME_IDX} is_key={cam_data[CAMERA]["is_key"]}')

# Show primary camera image
fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.imshow(cam_data[CAMERA]['img'])
ax.set_title(f'{DATASET} — scene {SCENE_IDX} ({scene_name}), frame {FRAME_IDX} — {CAMERA}')
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Raw LiDAR load — single sweep, all points

**Rule:** no ground filter, no visibility filter, no ego-body filter for the ray casting step.
We want every beam the sensor fired (§29, §6).

We also extract the **sensor origin** in ego frame (`t_l2e`), which is the starting point of
every ray.

In [ ]:
# ── Find anchor LiDAR token ───────────────────────────────────────────────────
cam_sd_token = cam_data[CAMERA]['sd_token']
cam_sd       = nusc.get('sample_data', cam_sd_token)
cam_ts       = cam_sd['timestamp']

if cam_sd['is_key_frame']:
    _sample     = nusc.get('sample', cam_sd['sample_token'])
    lidar_token = _sample['data']['LIDAR_TOP']
else:
    # Non-keyframe: find the LIDAR_TOP sweep closest in time
    lidar_token = min(
        (sd for sd in nusc.sample_data if sd['channel'] == 'LIDAR_TOP'),
        key=lambda sd: abs(sd['timestamp'] - cam_ts),
    )['token']

# ── Load the sweep (single sweep — no aggregation per §25.2) ─────────────────
lid_sd   = nusc.get('sample_data', lidar_token)
lid_cal  = nusc.get('calibrated_sensor', lid_sd['calibrated_sensor_token'])
lid_pose = nusc.get('ego_pose', lid_sd['ego_pose_token'])
lid_path = nusc.get_sample_data_path(lidar_token)

# NOTE (fixed 2026-08-31): R_e2g/t_e2g were anchored on the CAMERA's ego pose (cell §2),
# but LiDAR points below are only sensor->ego rigid-transformed (no pose correction), so
# they're implicitly expressed in the ego frame AT THE LIDAR's OWN timestamp. Camera and
# LiDAR keyframes can differ by tens of ms -- confirmed on this dataset: up to ~28ms, i.e.
# ~21cm of ego motion -- which was silently misaligning every GT/pred box overlay (and
# every OBB free% query downstream) from the LiDAR data by that amount. Overwrite with
# the LiDAR's own ego pose, the correct anchor for everything built in this frame.
R_e2g = Quaternion(lid_pose['rotation']).rotation_matrix.astype(np.float64)
t_e2g = np.array(lid_pose['translation'], dtype=np.float64)

# Sensor→ego rotation and translation
R_l2e = Quaternion(lid_cal['rotation']).rotation_matrix.astype(np.float64)
t_l2e = np.array(lid_cal['translation'], dtype=np.float64)

# Sensor origin in ego frame (this is the start of every ray)
sensor_origin = t_l2e.copy()   # (3,) in ego frame

# Load raw points from binary file (x, y, z, intensity, ring)
pts_raw = np.fromfile(lid_path, dtype=np.float32).reshape(-1, 5)[:, :3].astype(np.float64)

# Transform to ego frame: pts_ego = R_l2e @ pts_raw.T + t_l2e
# This is the full unfiltered sweep — ALL beams, including ground returns.
pts_ego = (R_l2e @ pts_raw.T).T + t_l2e

# Range in ego frame (for filtering by distance)
ranges = np.linalg.norm(pts_ego - sensor_origin, axis=1)

# Apply only range limits — no ground filter, no ego-body filter
range_mask = (ranges >= RAY_MIN_RANGE) & (ranges <= RAY_MAX_RANGE)
pts_ego_cast = pts_ego[range_mask].astype(np.float32)
ranges_cast  = ranges[range_mask].astype(np.float32)

dt_ms = abs(cam_ts - lid_sd['timestamp']) / 1_000.0
print(f'LiDAR file  : {lid_path}')
print(f'Δt (lidar–cam): {dt_ms:.1f} ms')
print(f'Raw points  : {len(pts_raw):,}')
print(f'After range [{RAY_MIN_RANGE}, {RAY_MAX_RANGE}] m filter: {len(pts_ego_cast):,} points')
print(f'Sensor origin in ego frame: {sensor_origin}')
print(f'Range stats: min={ranges_cast.min():.2f} m  '
      f'median={np.median(ranges_cast):.2f} m  '
      f'max={ranges_cast.max():.2f} m')

## 4. Amanatides & Woo ray casting

**Algorithm:** Amanatides & Woo (1987), "A Fast Voxel Traversal Algorithm for Ray Tracing."

For each LiDAR beam:
1. Compute direction from sensor origin to return point
2. March through voxels using the A&W step — advancing to the nearest voxel
   boundary at each step (O(N) where N = number of voxels along ray, not O(grid_size))
3. Each traversed voxel gets `LO_FREE` added (negative → more free)
4. The voxel containing the return point gets `LO_OCC` added (positive → more occupied)

Log-odds are clamped to `[LO_MIN, LO_MAX]` to prevent saturation.

This is implemented **without any library** — the traversal logic is visible and modifiable (§27 Phase 0.1).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Amanatides & Woo fast voxel traversal
# Reference: Amanatides J, Woo A. "A fast voxel traversal algorithm for
#            ray tracing." Eurographics 1987.
# ══════════════════════════════════════════════════════════════════════════════

if HAS_NUMBA:
    @numba.njit(cache=True)
    def _cast_ray_amanatides_woo(
            log_odds,           # float32 array [NX, NY, NZ]
            ox, oy, oz,         # ray origin (ego frame, meters)
            dx, dy, dz,         # normalized ray direction
            t_max,              # ray length (meters) — return point at origin + t_max*dir
            gox, goy, goz,      # grid origin (meters)
            vs,                 # voxel size (meters)
            NX, NY, NZ,         # grid dimensions (voxels)
            lo_free, lo_occ,    # log-odds increments
            lo_min, lo_max):    # clamping limits
        """
        Cast a single ray using Amanatides & Woo traversal.
        Marks traversed voxels FREE and the return voxel OCCUPIED.
        """
        INF = 1e30

        # ── Starting voxel index ──────────────────────────────────────────────
        ix = int((ox - gox) / vs)
        iy = int((oy - goy) / vs)
        iz = int((oz - goz) / vs)

        # Exit immediately if origin is outside the grid
        if ix < 0 or ix >= NX or iy < 0 or iy >= NY or iz < 0 or iz >= NZ:
            return

        # ── Step direction per axis ───────────────────────────────────────────
        sx = 1 if dx > 0.0 else (-1 if dx < 0.0 else 0)
        sy = 1 if dy > 0.0 else (-1 if dy < 0.0 else 0)
        sz = 1 if dz > 0.0 else (-1 if dz < 0.0 else 0)

        # ── tDelta: t increment to traverse exactly one voxel in each axis ───
        tdx = vs / abs(dx) if abs(dx) > 1e-9 else INF
        tdy = vs / abs(dy) if abs(dy) > 1e-9 else INF
        tdz = vs / abs(dz) if abs(dz) > 1e-9 else INF

        # ── tMax: t at which ray first crosses boundary in each axis ──────────
        # Boundary of current voxel in +step direction
        if sx > 0:
            tmx = (gox + (ix + 1) * vs - ox) / dx
        elif sx < 0:
            tmx = (gox + ix * vs - ox) / dx
        else:
            tmx = INF

        if sy > 0:
            tmy = (goy + (iy + 1) * vs - oy) / dy
        elif sy < 0:
            tmy = (goy + iy * vs - oy) / dy
        else:
            tmy = INF

        if sz > 0:
            tmz = (goz + (iz + 1) * vs - oz) / dz
        elif sz < 0:
            tmz = (goz + iz * vs - oz) / dz
        else:
            tmz = INF

        # ── March along ray ───────────────────────────────────────────────────
        while True:
            # Next voxel boundary crossing (smallest tMax)
            if tmx <= tmy:
                if tmx <= tmz:
                    t_cross = tmx
                else:
                    t_cross = tmz
            else:
                if tmy <= tmz:
                    t_cross = tmy
                else:
                    t_cross = tmz

            if t_cross >= t_max:
                # Return point is inside current voxel → mark OCCUPIED
                if 0 <= ix < NX and 0 <= iy < NY and 0 <= iz < NZ:
                    v = log_odds[ix, iy, iz] + lo_occ
                    log_odds[ix, iy, iz] = v if v < lo_max else lo_max
                return

            # Current voxel is on the free-space path → mark FREE
            if 0 <= ix < NX and 0 <= iy < NY and 0 <= iz < NZ:
                v = log_odds[ix, iy, iz] + lo_free
                log_odds[ix, iy, iz] = v if v > lo_min else lo_min

            # Advance to next voxel
            if tmx <= tmy:
                if tmx <= tmz:
                    ix += sx
                    tmx += tdx
                else:
                    iz += sz
                    tmz += tdz
            else:
                if tmy <= tmz:
                    iy += sy
                    tmy += tdy
                else:
                    iz += sz
                    tmz += tdz

            # Out-of-grid exit
            if ix < 0 or ix >= NX or iy < 0 or iy >= NY or iz < 0 or iz >= NZ:
                return


    @numba.njit(cache=True)
    def _raytrace_all_numba(
            log_odds,       # float32 [NX, NY, NZ]
            pts_ego,        # float32 [N, 3] — return points in ego frame
            ox, oy, oz,     # sensor origin in ego frame
            gox, goy, goz,  # grid origin
            vs, NX, NY, NZ,
            lo_free, lo_occ, lo_min, lo_max):
        """Serial loop over all rays. Serial is correct (no race conditions)."""
        N = pts_ego.shape[0]
        for i in range(N):
            px, py, pz = pts_ego[i, 0], pts_ego[i, 1], pts_ego[i, 2]
            rdx = px - ox
            rdy = py - oy
            rdz = pz - oz
            t_max = (rdx * rdx + rdy * rdy + rdz * rdz) ** 0.5
            if t_max < 1e-6:
                continue
            dx = rdx / t_max
            dy = rdy / t_max
            dz = rdz / t_max
            _cast_ray_amanatides_woo(
                log_odds,
                ox, oy, oz,
                dx, dy, dz, t_max,
                gox, goy, goz, vs,
                NX, NY, NZ,
                lo_free, lo_occ, lo_min, lo_max)


    def raytrace(log_odds, pts_ego, sensor_origin, grid_origin, voxel_size,
                 lo_free, lo_occ, lo_min, lo_max):
        """Entry point: wraps the numba kernel."""
        ox, oy, oz   = float(sensor_origin[0]), float(sensor_origin[1]), float(sensor_origin[2])
        gox, goy, goz = float(grid_origin[0]),  float(grid_origin[1]),  float(grid_origin[2])
        NX, NY, NZ   = log_odds.shape
        _raytrace_all_numba(
            log_odds, pts_ego.astype(np.float32),
            ox, oy, oz, gox, goy, goz,
            float(voxel_size), NX, NY, NZ,
            float(lo_free), float(lo_occ), float(lo_min), float(lo_max))


else:
    # ── numpy DDA fallback (transparent but slower) ───────────────────────────
    # Uses the same A&W logic in Python — correct but ~50× slower than numba.

    def _cast_ray_numpy(log_odds, origin, point, grid_origin, voxel_size,
                        NX, NY, NZ, lo_free, lo_occ, lo_min, lo_max):
        """Single-ray A&W traversal in pure Python (fallback when numba unavailable)."""
        INF = 1e30
        gox, goy, goz = grid_origin
        vs = voxel_size
        ox, oy, oz = origin
        rdx, rdy, rdz = point - origin
        t_max = float(np.sqrt(rdx**2 + rdy**2 + rdz**2))
        if t_max < 1e-6:
            return
        dx, dy, dz = rdx/t_max, rdy/t_max, rdz/t_max

        ix = int((ox - gox) / vs)
        iy = int((oy - goy) / vs)
        iz = int((oz - goz) / vs)
        if not (0 <= ix < NX and 0 <= iy < NY and 0 <= iz < NZ):
            return

        sx = 1 if dx > 0 else (-1 if dx < 0 else 0)
        sy = 1 if dy > 0 else (-1 if dy < 0 else 0)
        sz = 1 if dz > 0 else (-1 if dz < 0 else 0)
        tdx = vs / abs(dx) if abs(dx) > 1e-9 else INF
        tdy = vs / abs(dy) if abs(dy) > 1e-9 else INF
        tdz = vs / abs(dz) if abs(dz) > 1e-9 else INF
        tmx = (gox + (ix+1)*vs - ox) / dx if sx > 0 else ((gox + ix*vs - ox) / dx if sx < 0 else INF)
        tmy = (goy + (iy+1)*vs - oy) / dy if sy > 0 else ((goy + iy*vs - oy) / dy if sy < 0 else INF)
        tmz = (goz + (iz+1)*vs - oz) / dz if sz > 0 else ((goz + iz*vs - oz) / dz if sz < 0 else INF)

        while True:
            if tmx <= tmy:
                t_cross = tmx if tmx <= tmz else tmz
            else:
                t_cross = tmy if tmy <= tmz else tmz

            if t_cross >= t_max:
                if 0 <= ix < NX and 0 <= iy < NY and 0 <= iz < NZ:
                    log_odds[ix, iy, iz] = np.clip(log_odds[ix,iy,iz] + lo_occ, lo_min, lo_max)
                return

            if 0 <= ix < NX and 0 <= iy < NY and 0 <= iz < NZ:
                log_odds[ix, iy, iz] = np.clip(log_odds[ix,iy,iz] + lo_free, lo_min, lo_max)

            if tmx <= tmy:
                if tmx <= tmz:
                    ix += sx; tmx += tdx
                else:
                    iz += sz; tmz += tdz
            else:
                if tmy <= tmz:
                    iy += sy; tmy += tdy
                else:
                    iz += sz; tmz += tdz

            if not (0 <= ix < NX and 0 <= iy < NY and 0 <= iz < NZ):
                return

    def raytrace(log_odds, pts_ego, sensor_origin, grid_origin, voxel_size,
                 lo_free, lo_occ, lo_min, lo_max):
        """Entry point: numpy fallback — loops over rays in Python."""
        NX, NY, NZ = log_odds.shape
        print(f'  (numpy fallback: casting {len(pts_ego):,} rays — may take a minute)')
        for i, pt in enumerate(pts_ego):
            if i % 10_000 == 0 and i > 0:
                print(f'  {i:,}/{len(pts_ego):,} rays...')
            _cast_ray_numpy(log_odds, sensor_origin, pt, grid_origin, voxel_size,
                            NX, NY, NZ, lo_free, lo_occ, lo_min, lo_max)


print('Ray caster defined.')

## 5. Build the free-space map

Allocate the log-odds voxel grid and run ray casting.

In [ ]:
import time

# ── Voxel grid dimensions ──────────────────────────────────────────────────────
NX = int((GRID_X_MAX - GRID_X_MIN) / VOXEL_SIZE)
NY = int((GRID_Y_MAX - GRID_Y_MIN) / VOXEL_SIZE)
NZ = int((GRID_Z_MAX - GRID_Z_MIN) / VOXEL_SIZE)
grid_origin = np.array([GRID_X_MIN, GRID_Y_MIN, GRID_Z_MIN], dtype=np.float32)

print(f'Grid: {NX} × {NY} × {NZ}  =  {NX*NY*NZ/1e6:.1f} M voxels  '
      f'({NX*NY*NZ*4/1024**2:.1f} MB)')

# Initialise log-odds to 0 (UNKNOWN = uninformed prior)
log_odds = np.zeros((NX, NY, NZ), dtype=np.float32)

# ── Ray casting ────────────────────────────────────────────────────────────────
so = sensor_origin.astype(np.float32)

if HAS_NUMBA:
    print('Compiling JIT kernel (first call only)...')
    # Warm-up with one point to trigger compilation
    _warm = pts_ego_cast[:1].copy()
    raytrace(log_odds, _warm, so, grid_origin, VOXEL_SIZE,
             LO_FREE, LO_OCC, LO_MIN, LO_MAX)
    log_odds[:] = 0.0  # reset after warmup
    print('Compiled. Casting all rays...')

t0 = time.time()
raytrace(log_odds, pts_ego_cast, so, grid_origin, VOXEL_SIZE,
         LO_FREE, LO_OCC, LO_MIN, LO_MAX)
elapsed = time.time() - t0

# ── Summary statistics ─────────────────────────────────────────────────────────
# Chunked along X (fixed 2026-08-31): a direct (log_odds == 0.0).sum() on the full
# grid allocates a full-size 610 MB bool temporary for EACH of these three lines.
# On ECP that's on top of an already ~7.4 GB NuScenes(v1.0-trainval) index and the
# 2.4 GB grid itself, which is enough to blow the kernel's memory guard (see §0).
# Chunking keeps the peak temporary at ~1/20th the size with no accuracy cost.
n_total = NX * NY * NZ
n_unknown = n_free = n_occupied = 0
_chunk = max(1, NX // 20)
for _i0 in range(0, NX, _chunk):
    _sub = log_odds[_i0:_i0 + _chunk]
    n_unknown  += int((_sub == 0.0).sum())
    n_free     += int((_sub < THRESHOLD_FREE).sum())
    n_occupied += int((_sub > THRESHOLD_OCC).sum())
del _sub
n_uncertain = n_total - n_unknown - n_free - n_occupied

print(f'\nRay casting: {len(pts_ego_cast):,} rays in {elapsed:.2f} s '
      f'({len(pts_ego_cast)/elapsed:.0f} rays/s)')
print(f'Grid population:')
print(f'  UNKNOWN  (no beams) : {n_unknown:>10,}  ({100*n_unknown/n_total:.1f}%)')
print(f'  UNCERTAIN (between)  : {n_uncertain:>10,}  ({100*n_uncertain/n_total:.1f}%)')
print(f'  FREE (lo < {THRESHOLD_FREE})   : {n_free:>10,}  ({100*n_free/n_total:.1f}%)')
print(f'  OCCUPIED (lo > {THRESHOLD_OCC}) : {n_occupied:>10,}  ({100*n_occupied/n_total:.1f}%)')

## 6. Load GT boxes

GT annotations exist on keyframes. Walk back to the nearest keyframe if needed.

In [ ]:
# ── Load GT annotations from the nearest keyframe ────────────────────────────
# Returns a list of dicts with 3D box info in ego frame.

def load_gt_boxes_ego(nusc, cam_sd_token, R_e2g, t_e2g):
    """Walk back through sample_data to find the nearest keyframe with GT."""
    gt_boxes = []
    token = cam_sd_token
    while token:
        sd = nusc.get('sample_data', token)
        if sd['is_key_frame']:
            sample = nusc.get('sample', sd['sample_token'])
            for ann_token in sample['anns']:
                ann        = nusc.get('sample_annotation', ann_token)
                center_g   = np.array(ann['translation'], dtype=np.float64)
                size       = np.array(ann['size'],        dtype=np.float64)  # [W, L, H]
                rot_g      = Quaternion(ann['rotation'])

                # Global → anchor ego frame
                center_e   = R_e2g.T @ (center_g - t_e2g)
                R_box_e    = R_e2g.T @ rot_g.rotation_matrix
                yaw        = float(np.arctan2(R_box_e[1, 0], R_box_e[0, 0]))

                # Build 8 corners in ego frame
                # NuScenes size = [width, length, height] = [Y, X, Z] in object frame
                half_w, half_l, half_h = size[0]/2, size[1]/2, size[2]/2
                # Local corners (object frame: +X forward, +Y left, +Z up)
                cx = np.array([ half_l, half_l, -half_l, -half_l,
                                 half_l, half_l, -half_l, -half_l])
                cy = np.array([ half_w,-half_w, -half_w,  half_w,
                                 half_w,-half_w, -half_w,  half_w])
                cz = np.array([-half_h,-half_h, -half_h, -half_h,
                                 half_h, half_h,  half_h,  half_h])
                corners_local = np.stack([cx, cy, cz], axis=1)  # (8, 3)

                # Rotate by yaw (Z-axis rotation) and translate to ego center
                cos_y, sin_y = np.cos(yaw), np.sin(yaw)
                R_yaw = np.array([[cos_y, -sin_y, 0],
                                   [sin_y,  cos_y, 0],
                                   [    0,      0, 1]])
                corners_ego = (R_yaw @ corners_local.T).T + center_e

                # 2D BEV corners (first 4)
                corners_2d_bev = corners_ego[:4, :2]   # bottom face

                gt_boxes.append({
                    'center_e'   : center_e,
                    'size'       : size,          # [W, L, H]
                    'yaw'        : yaw,
                    'corners_ego': corners_ego,   # (8, 3)
                    'corners_2d' : corners_2d_bev,
                    'category'   : ann['category_name'],
                })
            print(f'GT: {len(gt_boxes)} annotation(s) from keyframe '
                  f'{sd["sample_token"][:8]}...')
            return gt_boxes
        token = sd['prev'] if sd['prev'] else ''
    print('GT: no preceding keyframe found.')
    return gt_boxes


gt_boxes = load_gt_boxes_ego(nusc, cam_sd_token, R_e2g, t_e2g)

# Print summary
from collections import Counter
cats = Counter(b['category'] for b in gt_boxes)
for cat, cnt in cats.most_common():
    print(f'  {cat}: {cnt}')

## 7. BEV visualisation

Bird's-Eye View slice at ~1 m height showing:
- **UNKNOWN** → grey  
- **FREE** (log-odds < threshold) → blue  
- **OCCUPIED** (log-odds > threshold) → red  
- GT boxes → yellow outlines  
- Camera FOV frustums → green wedges

In [ ]:
# ── Select BEV Z slice ────────────────────────────────────────────────────
z_slice_center = BEV_Z_SLICE_HEIGHT
iz_center = int((z_slice_center - GRID_Z_MIN) / VOXEL_SIZE)
iz_half   = max(1, int(BEV_Z_HALF_WIDTH / VOXEL_SIZE))
iz_lo     = max(0, iz_center - iz_half)
iz_hi     = min(NZ, iz_center + iz_half + 1)
# NOTE (fixed 2026-08-31): a plain .max(axis=2) over signed log-odds is wrong here --
# untouched voxels are exactly 0, free voxels are negative, and 0 > any negative value,
# so a single untouched voxel anywhere in the Z-band masks real free-space evidence from
# every other layer in that column. Use an any()-based, occupied-takes-priority reduction
# instead (matches how the actual OBB queries treat 3D evidence).
_occ_mask  = (log_odds[:, :, iz_lo:iz_hi] > THRESHOLD_OCC).any(axis=2)
_free_mask = (log_odds[:, :, iz_lo:iz_hi] < THRESHOLD_FREE).any(axis=2) & ~_occ_mask   # (NX, NY)
print(f'BEV Z slice: iz {iz_lo}-{iz_hi}  '
      f'(Z = [{GRID_Z_MIN + iz_lo*VOXEL_SIZE:.1f}, {GRID_Z_MIN + iz_hi*VOXEL_SIZE:.1f}] m)')

# ── Build RGB image ────────────────────────────────────────────────────
# Standard autonomous-driving BEV convention:
#   row 0 (top)   = ego +X (forward)    -- achieved by [::-1] on X axis
#   col 0 (left)  = ego +Y (left side)  -- achieved by [::-1] on Y axis
# log_odds[ix,iy] -> image[NX-1-ix, NY-1-iy]  =>  bev_rgb[::-1, ::-1]
bev_rgb = np.full((NX, NY, 3), 200, dtype=np.uint8)
bev_rgb[_free_mask] = [100, 149, 237]   # blue  = FREE
bev_rgb[_occ_mask]  = [220,  50,  50]   # red   = OCCUPIED
bev_display = bev_rgb[::-1, ::-1, :]   # (NX, NY, 3) -- no transpose needed

# ── Pixel coordinate helper ───────────────────────────────────────────────
def ego_xy_to_px(x, y):
    # ego (X, Y) meters -> (col, row) in bev_display (full, before crop)
    # forward (+X) -> row 0 (top);  left (+Y) -> col 0 (left)
    row = NX - 1 - (x - GRID_X_MIN) / VOXEL_SIZE
    col = NY - 1 - (y - GRID_Y_MIN) / VOXEL_SIZE
    return float(col), float(row)

# ── Crop to BEV_MAX_RANGE_PLOT ───────────────────────────────────────────────
r_px = int(BEV_MAX_RANGE_PLOT / VOXEL_SIZE)
ego_col_f, ego_row_f = ego_xy_to_px(0.0, 0.0)
ego_col, ego_row = int(round(ego_col_f)), int(round(ego_row_f))
r_lo = max(0, ego_row - r_px);   r_hi = min(NX, ego_row + r_px)
c_lo = max(0, ego_col - r_px);   c_hi = min(NY, ego_col + r_px)
bev_crop = bev_display[r_lo:r_hi, c_lo:c_hi]
orig_row = ego_row - r_lo
orig_col = ego_col - c_lo

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(bev_crop, origin='upper', interpolation='nearest')
# origin='upper': row 0 at top = ego forward. No ax.invert_yaxis() needed.

ax.plot(orig_col, orig_row, 'w*', markersize=12, zorder=5)
ax.annotate('ego', (orig_col + 3, orig_row - 3), color='white', fontsize=8, zorder=5)

# ── GT boxes ──────────────────────────────────────────────────────────────────────
GT_COLORS = {
    'human.pedestrian.adult':      'yellow',
    'human.pedestrian.child':      'yellow',
    'human.pedestrian.wheelchair': 'yellow',
    'vehicle.car':                 'lime',
    'vehicle.bicycle':             'cyan',
    'vehicle.motorcycle':          'orange',
    'vehicle.bus.bendy':           'magenta',
    'vehicle.bus.rigid':           'magenta',
    'vehicle.truck':               'violet',
    'vehicle.construction':        'pink',
    'vehicle.trailer':             'khaki',
}

def draw_gt_box_bev(ax, corners_2d, category, c_lo, r_lo):
    col_gt = GT_COLORS.get(category, 'yellow')
    pts = np.array([ego_xy_to_px(c[0], c[1]) for c in corners_2d])  # (4,2) col,row
    pts[:, 0] -= c_lo
    pts[:, 1] -= r_lo
    poly = plt.Polygon(pts, fill=False, edgecolor=col_gt, linewidth=1.5, zorder=4)
    ax.add_patch(poly)

for gb in gt_boxes:
    dist = float(np.linalg.norm(gb['center_e'][:2]))
    if dist > BEV_MAX_RANGE_PLOT:
        continue
    draw_gt_box_bev(ax, gb['corners_2d'], gb['category'], c_lo, r_lo)

# ── Camera FOV frustums ──────────────────────────────────────────────────────
# Use left-edge (u=0) and right-edge (u=W) rays at vertical image centre.
# These give the correct horizontal FOV boundaries in BEV.
FRUSTUM_RANGE = min(BEV_MAX_RANGE_PLOT * 0.9, 38.0)
FOV_COLORS = ['limegreen', 'springgreen', 'darkgreen', 'teal', 'darkcyan', 'mediumaquamarine']

for idx_cam, cam in enumerate(CAMERAS):
    cd = cam_data[cam]
    K_c, R_c2e_c, t_c2e_c = cd['K'], cd['R_c2e'], cd['t_c2e']
    H_c, W_c = cd['H'], cd['W']
    fx_c, fy_c, cx_c, cy_c = K_c[0,0], K_c[1,1], K_c[0,2], K_c[1,2]

    # Left and right FOV boundary rays in camera frame
    left_cam  = np.array([(0   - cx_c)/fx_c,  (H_c/2 - cy_c)/fy_c,  1.0])
    right_cam = np.array([(W_c - cx_c)/fx_c,  (H_c/2 - cy_c)/fy_c,  1.0])

    fc = FOV_COLORS[idx_cam % len(FOV_COLORS)]
    cam_col_c, cam_row_c = ego_xy_to_px(t_c2e_c[0], t_c2e_c[1])
    cam_col_c -= c_lo;  cam_row_c -= r_lo

    for d_cam in [left_cam, right_cam]:
        d_ego = R_c2e_c @ d_cam
        d_norm = d_ego / max(np.linalg.norm(d_ego), 1e-9)
        far_pt = t_c2e_c + d_norm * FRUSTUM_RANGE
        far_col, far_row = ego_xy_to_px(far_pt[0], far_pt[1])
        ax.plot([cam_col_c, far_col - c_lo],
                [cam_row_c, far_row - r_lo],
                color=fc, linewidth=0.9, alpha=0.8, zorder=3)
    ax.plot(cam_col_c, cam_row_c, 's', color=fc, markersize=5, zorder=5)

# ── Legend + labels ────────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(color=[c/255 for c in [100,149,237]], label='Free (certified empty)'),
    mpatches.Patch(color=[c/255 for c in [220, 50, 50]], label='Occupied (LiDAR return)'),
    mpatches.Patch(color=[c/255 for c in [200,200,200]], label='Unknown'),
    mpatches.Patch(color='yellow',    label='GT box'),
    mpatches.Patch(color='limegreen', label='Camera FOV'),
]
ax.legend(handles=legend_patches, loc='upper right', fontsize=8)
ax.set_title(
    f'{DATASET} — scene {SCENE_IDX}, frame {FRAME_IDX} | '
    f'BEV Z={z_slice_center:.1f}m | voxel={VOXEL_SIZE}m | {LIDAR_LINES}-beam LiDAR'
)
# X-axis = Y-ego direction (col 0 = left = +Y_MAX)
# Y-axis = X-ego direction (row 0 = forward = +X_MAX)
ax.set_xlabel('← +Y (left)                              -Y (right) →')
ax.set_ylabel('+X (forward) ↑              ↓ -X (back)')
_tick_m  = 10.0
_tick_px = int(_tick_m / VOXEL_SIZE)
x_ticks = range(0, bev_crop.shape[1], _tick_px)
ax.set_xticks(list(x_ticks))
ax.set_xticklabels(
    [f'{GRID_Y_MAX - (c_lo + i)*VOXEL_SIZE:.0f}m' for i in x_ticks], fontsize=7)
y_ticks = range(0, bev_crop.shape[0], _tick_px)
ax.set_yticks(list(y_ticks))
ax.set_yticklabels(
    [f'{GRID_X_MAX - (r_lo + i)*VOXEL_SIZE:.0f}m' for i in y_ticks], fontsize=7)
plt.tight_layout()

out_path = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'phase0_occupancy_map_example_{DATASET}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()


### 7.1 Side profile through the closest GT car

Cuts a 2D plane through the box's own length-height axis, at width=0 (through its
centreline) -- a true side-view profile regardless of yaw. An axis-aligned X-Z or
Y-Z slice (like §10 above) would only show this correctly if the car happened to be
driving exactly along the X or Y axis; this one rotates with the box instead.

In [ ]:
# ── Side profile through the closest 'car' GT box (width=0 centreline cut) ──────
_car_boxes = [b for b in gt_boxes if b['category'] == 'car']
if not _car_boxes:
    print("No 'car' GT boxes in this frame -- pick a different frame or category.")
else:
    _target = min(_car_boxes, key=lambda b: np.linalg.norm(b['center_e'][:2]))
    _cx, _cy, _cz = _target['center_e']
    _half_w, _half_l, _half_h = (_target['size'][0] / 2, _target['size'][1] / 2,
                                  _target['size'][2] / 2)
    _yaw = _target['yaw']
    _dir = np.array([np.cos(_yaw), np.sin(_yaw)])   # unit vector along car length, world XY
    _dist = float(np.linalg.norm(_target['center_e'][:2]))
    print(f'Closest car: range={_dist:.1f}m  size(W,L,H)=({_target["size"][0]:.2f},'
          f'{_target["size"][1]:.2f},{_target["size"][2]:.2f})  yaw={np.degrees(_yaw):.0f}deg')

    _margin_u = 2.0   # metres of context beyond the box on each end, along its length
    _u = np.arange(-_half_l - _margin_u, _half_l + _margin_u, VOXEL_SIZE)
    _v_lo, _v_hi = _cz - _half_h - 1.0, _cz + _half_h + 1.5
    _v = np.arange(_v_lo, _v_hi, VOXEL_SIZE)
    _uu, _vv = np.meshgrid(_u, _v, indexing='ij')   # (len(_u), len(_v))

    # World coords of the sampling plane: centre + u*length_dir (width fixed at 0) + v*Z_hat
    _wx = _cx + _uu * _dir[0]
    _wy = _cy + _uu * _dir[1]
    _wz = _vv

    _ix = np.clip(((_wx - GRID_X_MIN) / VOXEL_SIZE).astype(np.int64), 0, NX - 1)
    _iy = np.clip(((_wy - GRID_Y_MIN) / VOXEL_SIZE).astype(np.int64), 0, NY - 1)
    _iz = np.clip(((_wz - GRID_Z_MIN) / VOXEL_SIZE).astype(np.int64), 0, NZ - 1)
    _lo_vals = log_odds[_ix, _iy, _iz]   # (len(_u), len(_v)) -- single-voxel lookup per
                                          # sample point, no Z-band collapsing involved

    _occ_mask  = _lo_vals > THRESHOLD_OCC
    _free_mask = _lo_vals < THRESHOLD_FREE

    _img = np.full((len(_v), len(_u), 3), 200, dtype=np.uint8)   # rows=height, cols=length
    _img[_free_mask.T] = [100, 149, 237]
    _img[_occ_mask.T]  = [220, 50, 50]

    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    ax.imshow(_img, origin='lower', extent=[_u[0], _u[-1], _v[0], _v[-1]],
              aspect='auto', interpolation='nearest')

    ax.axhline(0.0, color='yellow', linewidth=1, linestyle='--')
    ax.axhline(sensor_origin[2], color='white', linewidth=1, linestyle=':')
    _box_rect = plt.Rectangle((-_half_l, _cz - _half_h), 2 * _half_l, 2 * _half_h,
                               fill=False, edgecolor='lime', linewidth=1.8)
    ax.add_patch(_box_rect)

    from matplotlib.lines import Line2D
    legend_handles = [
        mpatches.Patch(color=[100/255, 149/255, 237/255], label='Free (certified empty)'),
        mpatches.Patch(color=[220/255,  50/255,  50/255], label='Occupied (LiDAR return)'),
        mpatches.Patch(color=[200/255, 200/255, 200/255], label='Unknown'),
        Line2D([0], [0], color='yellow', linewidth=1, linestyle='--', label='Z=0 (ground)'),
        Line2D([0], [0], color='white', linewidth=1, linestyle=':',
               label=f'Sensor Z={sensor_origin[2]:.2f}m'),
        Line2D([0], [0], color='lime', linewidth=1.8, label='GT box (this car)'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
    ax.set_xlabel('u — distance along car length axis [m]  (0 = box centre)')
    ax.set_ylabel('Z (height) [m]')
    ax.set_title(
        f'{DATASET} — side profile through closest car GT box '
        f'(range={_dist:.1f}m, yaw={np.degrees(_yaw):.0f}°)\n'
        f'Cut through box centreline (width=0), voxel={VOXEL_SIZE}m'
    )
    plt.tight_layout()

    out_path = (DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' /
                f'phase0_side_profile_closest_car_{DATASET}.png')
    plt.savefig(out_path, dpi=140, bbox_inches='tight')
    print(f'Saved: {out_path}')
    plt.show()


### 7.2 Interactive cross-section slider (steps through the car, nose to tail)

Complements §7.1: that plot fixed width=0 and swept the whole length axis in one
image. This one does the opposite -- freeze a single position `u` along the car's
length and show the **width × height** slice there, with a slider to move `u` from
the near end of the box to the far end. Reuses `log_odds` and the box parameters
computed in §7.1 -- run that cell first.


In [ ]:
# ── Interactive width×height cross-section, stepping along the car's length ─────
# Left panel: the §7.1 side profile (reused, not recomputed) with a moving cursor
# line showing where the current cut is. Right panel: the width×height slice there.
from ipywidgets import interact, FloatSlider

_width_dir = np.array([-np.sin(_yaw), np.cos(_yaw)])   # perpendicular to length, world XY
_margin_w = 1.0
_w_ax = np.arange(-_half_w - _margin_w, _half_w + _margin_w, VOXEL_SIZE)
_v_ax = np.arange(_v_lo, _v_hi, VOXEL_SIZE)   # same height range as §7.1
_ww, _vv2 = np.meshgrid(_w_ax, _v_ax, indexing='ij')

_u_lo, _u_hi = -_half_l - _margin_u, _half_l + _margin_u

def _plot_cross_section(u=0.0):
    wx = _cx + u * _dir[0] + _ww * _width_dir[0]
    wy = _cy + u * _dir[1] + _ww * _width_dir[1]
    wz = _vv2

    ix = np.clip(((wx - GRID_X_MIN) / VOXEL_SIZE).astype(np.int64), 0, NX - 1)
    iy = np.clip(((wy - GRID_Y_MIN) / VOXEL_SIZE).astype(np.int64), 0, NY - 1)
    iz = np.clip(((wz - GRID_Z_MIN) / VOXEL_SIZE).astype(np.int64), 0, NZ - 1)
    lo_vals = log_odds[ix, iy, iz]

    occ_mask  = lo_vals > THRESHOLD_OCC
    free_mask = lo_vals < THRESHOLD_FREE

    cross_img = np.full((len(_v_ax), len(_w_ax), 3), 200, dtype=np.uint8)
    cross_img[free_mask.T] = [100, 149, 237]
    cross_img[occ_mask.T]  = [220, 50, 50]

    fig, (ax_side, ax_cross) = plt.subplots(
        1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.4, 1]})

    # ── Left: the §7.1 side profile, reused as-is, with a cursor line at the current u ──
    ax_side.imshow(_img, origin='lower', extent=[_u[0], _u[-1], _v[0], _v[-1]],
                   aspect='auto', interpolation='nearest')
    ax_side.axhline(0.0, color='yellow', linewidth=1, linestyle='--')
    ax_side.axhline(sensor_origin[2], color='white', linewidth=1, linestyle=':')
    ax_side.add_patch(plt.Rectangle((-_half_l, _cz - _half_h), 2 * _half_l, 2 * _half_h,
                                     fill=False, edgecolor='lime', linewidth=1.8))
    ax_side.axvline(u, color='magenta', linewidth=2.2)
    ax_side.set_xlabel('u — distance along car length axis [m]')
    ax_side.set_ylabel('Z (height) [m]')
    ax_side.set_title('Side profile (§7.1) — magenta line = current cut')

    # ── Right: width×height cross-section at u ──────────────────────────────────────
    ax_cross.imshow(cross_img, origin='lower',
                     extent=[_w_ax[0], _w_ax[-1], _v_ax[0], _v_ax[-1]],
                     aspect='equal', interpolation='nearest')
    ax_cross.axhline(0.0, color='yellow', linewidth=1, linestyle='--')
    ax_cross.axhline(sensor_origin[2], color='white', linewidth=1, linestyle=':')
    in_box_length = abs(u) <= _half_l
    box_edge_color = 'lime' if in_box_length else 'gray'
    ax_cross.add_patch(plt.Rectangle(
        (-_half_w, _cz - _half_h), 2 * _half_w, 2 * _half_h,
        fill=False, edgecolor=box_edge_color, linewidth=1.8,
        linestyle='-' if in_box_length else '--'))
    ax_cross.set_xlabel('width axis [m]  (0 = box centreline)')
    ax_cross.set_ylabel('Z (height) [m]')
    _where = 'inside box length' if in_box_length else 'OUTSIDE box length (context only)'
    ax_cross.set_title(f'Cross-section at u={u:+.2f}m  ({_where})')

    fig.suptitle(f'{DATASET} — closest car, range={_dist:.1f}m, box length=±{_half_l:.2f}m',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

interact(_plot_cross_section,
         u=FloatSlider(min=_u_lo, max=_u_hi, step=VOXEL_SIZE * 4, value=0.0,
                        readout_format='.2f', description='u [m]', continuous_update=False))


### 7.3 Interactive slider, reversed (walk sideways across the car instead)

Mirror image of §7.2: there the slider moved lengthwise (`u`) and the side view was
the static reference while the width×height cross-section updated each step. Here
the slider moves sideways (`w`, left↔right across the car) and it's the **side view
that updates** at each width offset, while the width×height cross-section (fixed at
the box's longitudinal centre) stays static with a magenta line marking the current
`w`. Run §7.1 and §7.2 first -- this reuses variables from both.


In [ ]:
# ── Reversed interactive view: slider walks sideways (w), side view updates ─────
_static_u = 0.0   # reference cross-section position -- the box's longitudinal centre

def _cross_section_at(u_val):
    wx = _cx + u_val * _dir[0] + _ww * _width_dir[0]
    wy = _cy + u_val * _dir[1] + _ww * _width_dir[1]
    wz = _vv2
    ix = np.clip(((wx - GRID_X_MIN) / VOXEL_SIZE).astype(np.int64), 0, NX - 1)
    iy = np.clip(((wy - GRID_Y_MIN) / VOXEL_SIZE).astype(np.int64), 0, NY - 1)
    iz = np.clip(((wz - GRID_Z_MIN) / VOXEL_SIZE).astype(np.int64), 0, NZ - 1)
    lo_vals = log_odds[ix, iy, iz]
    occ_mask  = lo_vals > THRESHOLD_OCC
    free_mask = lo_vals < THRESHOLD_FREE
    img = np.full((len(_v_ax), len(_w_ax), 3), 200, dtype=np.uint8)
    img[free_mask.T] = [100, 149, 237]
    img[occ_mask.T]  = [220, 50, 50]
    return img

_static_cross_img = _cross_section_at(_static_u)   # computed once -- doesn't change with w

def _plot_side_at_width(w=0.0):
    wx = _cx + _uu * _dir[0] + w * _width_dir[0]
    wy = _cy + _uu * _dir[1] + w * _width_dir[1]
    wz = _vv
    ix = np.clip(((wx - GRID_X_MIN) / VOXEL_SIZE).astype(np.int64), 0, NX - 1)
    iy = np.clip(((wy - GRID_Y_MIN) / VOXEL_SIZE).astype(np.int64), 0, NY - 1)
    iz = np.clip(((wz - GRID_Z_MIN) / VOXEL_SIZE).astype(np.int64), 0, NZ - 1)
    lo_vals = log_odds[ix, iy, iz]
    occ_mask  = lo_vals > THRESHOLD_OCC
    free_mask = lo_vals < THRESHOLD_FREE
    side_img = np.full((len(_v), len(_u), 3), 200, dtype=np.uint8)
    side_img[free_mask.T] = [100, 149, 237]
    side_img[occ_mask.T]  = [220, 50, 50]

    fig, (ax_side, ax_cross) = plt.subplots(
        1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [1.4, 1]})

    # ── Left: side view at this width offset -- updates every step ──────────────────
    ax_side.imshow(side_img, origin='lower', extent=[_u[0], _u[-1], _v[0], _v[-1]],
                   aspect='auto', interpolation='nearest')
    ax_side.axhline(0.0, color='yellow', linewidth=1, linestyle='--')
    ax_side.axhline(sensor_origin[2], color='white', linewidth=1, linestyle=':')
    in_box_width = abs(w) <= _half_w
    edge_color = 'lime' if in_box_width else 'gray'
    ax_side.add_patch(plt.Rectangle(
        (-_half_l, _cz - _half_h), 2 * _half_l, 2 * _half_h,
        fill=False, edgecolor=edge_color, linewidth=1.8,
        linestyle='-' if in_box_width else '--'))
    ax_side.set_xlabel('u — distance along car length axis [m]')
    ax_side.set_ylabel('Z (height) [m]')
    _where = 'inside box width' if in_box_width else 'OUTSIDE box width (context only)'
    ax_side.set_title(f'Side view at width={w:+.2f}m  ({_where})')

    # ── Right: static cross-section at the box centre, magenta line = current w ─────
    ax_cross.imshow(_static_cross_img, origin='lower',
                     extent=[_w_ax[0], _w_ax[-1], _v_ax[0], _v_ax[-1]],
                     aspect='equal', interpolation='nearest')
    ax_cross.axhline(0.0, color='yellow', linewidth=1, linestyle='--')
    ax_cross.axhline(sensor_origin[2], color='white', linewidth=1, linestyle=':')
    ax_cross.add_patch(plt.Rectangle((-_half_w, _cz - _half_h), 2 * _half_w, 2 * _half_h,
                                     fill=False, edgecolor='lime', linewidth=1.8))
    ax_cross.axvline(w, color='magenta', linewidth=2.2)
    ax_cross.set_xlabel('width axis [m]  (0 = box centreline)')
    ax_cross.set_ylabel('Z (height) [m]')
    ax_cross.set_title(f'Cross-section at u={_static_u:+.2f}m — magenta line = current width')

    fig.suptitle(f'{DATASET} — closest car, range={_dist:.1f}m, box width=±{_half_w:.2f}m',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

interact(_plot_side_at_width,
         w=FloatSlider(min=-_half_w - _margin_w, max=_half_w + _margin_w,
                        step=VOXEL_SIZE * 4, value=0.0, readout_format='.2f',
                        description='w [m]', continuous_update=False))


## 8. Per-object free-space statistics

For each GT box: compute the fraction of voxels inside the box footprint that are
**FREE / OCCUPIED / UNKNOWN**.

This is the core diagnostic for Experiment A (§23, §28 item 0.3):
- If the predicted box extends into CERTIFIED FREE voxels → **overshoot** (inflation)
- If the GT box has large UNKNOWN interior → self-occlusion / far range
- If GT box is mostly FREE → the sensor proved the object isn't there (FP candidate or wrong GT)

In [ ]:
def query_box_freespace(log_odds, corners_ego, grid_origin, voxel_size,
                        threshold_free=THRESHOLD_FREE, threshold_occ=THRESHOLD_OCC):
    """
    Query the log-odds map inside a GT box and return voxel statistics.

    Uses the axis-aligned bounding box of the 8 corners as the query volume.
    (A tighter oriented-box query is possible but AABB is sufficient for Phase 0.)

    Returns dict with keys: n_total, n_free, n_occupied, n_unknown, n_uncertain,
    frac_free, frac_occupied, frac_unknown, lo_values.
    """
    gox, goy, goz = grid_origin
    vs = voxel_size
    NX_g, NY_g, NZ_g = log_odds.shape

    # AABB of the 8 corners
    x_min, y_min, z_min = corners_ego.min(axis=0)
    x_max, y_max, z_max = corners_ego.max(axis=0)

    ix_lo = max(0, int((x_min - gox) / vs))
    ix_hi = min(NX_g, int((x_max - gox) / vs) + 1)
    iy_lo = max(0, int((y_min - goy) / vs))
    iy_hi = min(NY_g, int((y_max - goy) / vs) + 1)
    iz_lo = max(0, int((z_min - goz) / vs))
    iz_hi = min(NZ_g, int((z_max - goz) / vs) + 1)

    if ix_lo >= ix_hi or iy_lo >= iy_hi or iz_lo >= iz_hi:
        return None   # box outside grid

    sub = log_odds[ix_lo:ix_hi, iy_lo:iy_hi, iz_lo:iz_hi]
    lo_values = sub.flatten()

    n_total    = len(lo_values)
    n_free     = int((lo_values < threshold_free).sum())
    n_occupied = int((lo_values > threshold_occ).sum())
    n_unknown  = int((lo_values == 0.0).sum())
    n_uncertain = n_total - n_free - n_occupied - n_unknown

    return {
        'n_total'    : n_total,
        'n_free'     : n_free,
        'n_occupied' : n_occupied,
        'n_unknown'  : n_unknown,
        'n_uncertain': n_uncertain,
        'frac_free'  : n_free / n_total,
        'frac_occ'   : n_occupied / n_total,
        'frac_unk'   : n_unknown / n_total,
        'lo_values'  : lo_values,
        'aabb_m'     : (x_min, x_max, y_min, y_max, z_min, z_max),
    }


# ── Run for all GT boxes ───────────────────────────────────────────────────────
print(f'\nFree-space statistics for {len(gt_boxes)} GT boxes:\n')
print(f'{"#":>3}  {"Category":<35}  {"Range":>7}  {"Free%":>6}  {"Occ%":>5}  {"Unk%":>5}  {"N vox":>7}')
print('─' * 80)

box_stats = []
for i, gb in enumerate(gt_boxes):
    dist = float(np.linalg.norm(gb['center_e'][:2]))
    stats = query_box_freespace(log_odds, gb['corners_ego'], grid_origin, VOXEL_SIZE)
    if stats is None:
        box_stats.append(None)
        print(f'{i:>3}  {gb["category"]:<35}  {dist:>6.1f}m  (outside grid)')
        continue
    box_stats.append(stats)
    print(f'{i:>3}  {gb["category"]:<35}  {dist:>6.1f}m  '
          f'{100*stats["frac_free"]:>5.1f}%  '
          f'{100*stats["frac_occ"]:>4.1f}%  '
          f'{100*stats["frac_unk"]:>4.1f}%  '
          f'{stats["n_total"]:>7,}')

## 9. Per-object visualisation: log-odds distribution

For the 5 closest GT boxes (most likely to have dense free-space evidence), plot
the histogram of log-odds values inside the box.

- A peak at **negative values** = the box interior is mostly certified free
- A peak at **positive values** = the box interior is mostly occupied (returns on surface)
- A spike at **0** = mostly unknown (no beams reached here)

In [ ]:
# Sort by range, take up to 5 closest with valid stats
valid_boxes = [(i, gb, bs) for i, (gb, bs) in enumerate(zip(gt_boxes, box_stats))
               if bs is not None]
valid_boxes.sort(key=lambda x: np.linalg.norm(x[1]['center_e'][:2]))
n_show = min(5, len(valid_boxes))

if n_show == 0:
    print('No valid GT boxes with stats to show.')
else:
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4), sharey=False)
    if n_show == 1:
        axes = [axes]

    for col_idx, (box_idx, gb, stats) in enumerate(valid_boxes[:n_show]):
        ax = axes[col_idx]
        lo_vals = stats['lo_values']
        dist = float(np.linalg.norm(gb['center_e'][:2]))
        cat  = gb['category'].split('.')[-1]   # short name

        # Histogram with threshold lines
        ax.hist(lo_vals, bins=50, range=(LO_MIN, LO_MAX),
                color='steelblue', alpha=0.7, edgecolor='none')
        ax.axvline(THRESHOLD_FREE, color='blue',  linewidth=1.5, linestyle='--', label=f'Free thr')
        ax.axvline(THRESHOLD_OCC,  color='red',   linewidth=1.5, linestyle='--', label=f'Occ thr')
        ax.axvline(0,              color='black', linewidth=0.8, linestyle=':',  label='Unknown')

        ax.set_title(f'#{box_idx} {cat}\n{dist:.1f} m | '
                     f'{100*stats["frac_free"]:.0f}%F {100*stats["frac_occ"]:.0f}%O '
                     f'{100*stats["frac_unk"]:.0f}%U',
                     fontsize=9)
        ax.set_xlabel('Log-odds')
        ax.set_ylabel('Voxel count' if col_idx == 0 else '')
        if col_idx == 0:
            ax.legend(fontsize=7)

    plt.suptitle(
        f'{DATASET} — {LIDAR_LINES}-beam — 5 closest GT objects (sorted by range)',
        y=1.02
    )
    plt.tight_layout()
    plt.show()

## 10. BEV vertical cross-section

Plot a vertical slice (X–Z plane through ego origin) to see how the free-space
map looks in the height dimension. This is useful for checking whether ground
returns are contributing free-space evidence above the road surface, as expected.

In [ ]:
# ── X-Z slice at Y = 0 (ego centreline) ─────────────────────────────────────────────
iy_center = int((0.0 - GRID_Y_MIN) / VOXEL_SIZE)
iy_band   = max(1, int(1.0 / VOXEL_SIZE))
iy_lo_xz  = max(0, iy_center - iy_band)
iy_hi_xz  = min(NY, iy_center + iy_band + 1)
# NOTE (fixed 2026-08-31): occupied-takes-priority any() reduction, not a raw .max()
# on signed log-odds -- see the note in cell 16 for why the raw max silently masks free
# evidence behind any untouched (0) voxel in the collapsed band.
_occ_xz  = (log_odds[:, iy_lo_xz:iy_hi_xz, :] > THRESHOLD_OCC).any(axis=1)   # (NX, NZ)
_free_xz = (log_odds[:, iy_lo_xz:iy_hi_xz, :] < THRESHOLD_FREE).any(axis=1) & ~_occ_xz

# image[row_z, col_x]: col = ix (forward left->right), row = NZ-1-iz (up at top)
_occ_xz_disp  = _occ_xz.T[::-1, :]   # (NZ, NX)
_free_xz_disp = _free_xz.T[::-1, :]  # (NZ, NX)
xz_rgb = np.full((*_occ_xz_disp.shape, 3), 200, dtype=np.uint8)
xz_rgb[_free_xz_disp] = [100, 149, 237]
xz_rgb[_occ_xz_disp]  = [220,  50,  50]

# Crop to visible X range
ix_ego    = int((0.0 - GRID_X_MIN) / VOXEL_SIZE)
x_px_half = int(BEV_MAX_RANGE_PLOT / VOXEL_SIZE)
c_lo_xz   = max(0, ix_ego - x_px_half // 2)
c_hi_xz   = min(NX, ix_ego + x_px_half)
xz_crop   = xz_rgb[:, c_lo_xz:c_hi_xz]

fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.imshow(xz_crop, origin='upper', interpolation='nearest', aspect='auto')

iz_ground = NZ - 1 - int((0.0 - GRID_Z_MIN) / VOXEL_SIZE)
iz_sensor = NZ - 1 - int((sensor_origin[2] - GRID_Z_MIN) / VOXEL_SIZE)
ax.axhline(iz_ground, color='yellow', linewidth=1, linestyle='--', label='Z=0 (ground)')
ax.axhline(iz_sensor, color='white',  linewidth=1, linestyle=':',
           label=f'Sensor Z={sensor_origin[2]:.2f}m')
ax.axvline(ix_ego - c_lo_xz, color='white', linewidth=0.8, linestyle='--',
           alpha=0.5, label='Ego X=0')

_tick_m_xz  = 10.0
_tick_px_xz = int(_tick_m_xz / VOXEL_SIZE)
x_ticks_xz = range(0, xz_crop.shape[1], _tick_px_xz)
ax.set_xticks(list(x_ticks_xz))
ax.set_xticklabels(
    [f'{GRID_X_MIN + (c_lo_xz + i)*VOXEL_SIZE:.0f}m' for i in x_ticks_xz], fontsize=7)
z_ticks = range(0, xz_crop.shape[0], _tick_px_xz)
ax.set_yticks(list(z_ticks))
ax.set_yticklabels(
    [f'{GRID_Z_MAX - i*VOXEL_SIZE:.0f}m' for i in z_ticks], fontsize=7)
ax.set_xlabel('X (ego forward) [m]')
ax.set_ylabel('Z (height) [m]  ↑ up')
ax.set_title(f'{DATASET} — X–Z cross-section (Y=0 ±1m)  |  {LIDAR_LINES}-beam LiDAR')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

## 11. Summary and notes for next steps

**Questions to answer from the visualisations above (§30 open questions):**

1. Do GT box interiors show significant FREE evidence at close range? → Overshoot potential
2. Do far GT boxes show mostly UNKNOWN interior? → Expected (self-occlusion shadow)
3. Is the 32-beam (nuScenes) free-space map visually sparse compared to 64-beam (ECP)?  
   → Set `DATASET = 'nuscenes_mini'` and re-run to compare
4. Do the camera FOV frustums align with the visible free-space structure?

**What comes next (§28 Phase 0, items 0.3–0.5):**
- 0.3: Experiment A — for each TP, check if the predicted box overshoots into CERTIFIED FREE voxels  
  (needs the pipeline OBB output, deferred until ECP GT frames are loaded)
- 0.4: Plot overshoot/undershoot per class and per §3 regime cell
- 0.5: Compare recall O3 vs O5 to decide base config

**Immediate next step:** provide the specific ECP scene/frame paths to load a real GT frame.

In [ ]:
# ── Quick sanity check: ray count per GT box vs. range ─────────────────────────
print('Sanity check: LiDAR points inside GT box footprints vs. range\n')
print(f'{"#":>3}  {"Category":<30}  {"Range":>7}  {"Occ vox":>7}  {"Free vox":>8}  {"Unk vox":>7}')
print('─' * 75)

for i, (gb, stats) in enumerate(zip(gt_boxes, box_stats)):
    if stats is None:
        continue
    dist = float(np.linalg.norm(gb['center_e'][:2]))
    cat  = gb['category'].split('.')[-1]
    print(f'{i:>3}  {cat:<30}  {dist:>6.1f}m  '
          f'{stats["n_occupied"]:>7,}  '
          f'{stats["n_free"]:>8,}  '
          f'{stats["n_unknown"]:>7,}')

print(f'\nRun with DATASET="nuscenes_mini" to compare 32-beam density.')

## 12. Experiment A — Free-space vs Predicted Boxes

**Goal (§28 Phase 0, item 0.3):** Load O3 pipeline predictions for this frame, transform them to ego frame, and count how many of the predicted-box voxels fall inside certified-free space.

A predicted box with ≥1 FREE voxel inside it has extended into volume that the sensor has *certified empty* — a hard geometric contradiction. This is the first quantitative evidence that the free-space map can catch overinflated predictions without any GT.

In [ ]:
import json, os

PRED_DIR_NUSCENES = ('/workspace/output'
                     '/nuscenes_mini/lidar_integration_objects_o3_body_b1_all_cameras')
PRED_DIR_ECP      = ('/workspace/output'
                     '/ecp/lidar_integration_objects_o3_body_b1_all_cameras')

if DATASET == 'nuscenes_mini':
    pred_file = os.path.join(PRED_DIR_NUSCENES, 'autolabel_nuscenes_mini_train_8class.json')
else:
    pred_file = os.path.join(PRED_DIR_ECP, 'autolabel_ecp_annotated_8class.json')

with open(pred_file) as _f:
    pred_data = json.load(_f)

# Find the keyframe sample token from the loaded camera sample_data
_tok = cam_sd_token
while _tok:
    _sd = nusc.get('sample_data', _tok)
    if _sd['is_key_frame']:
        _sample_token = _sd['sample_token']
        break
    _tok = _sd['prev'] if _sd['prev'] else ''

print(f'Sample token : {_sample_token}')
if _sample_token not in pred_data['results']:
    print('WARNING: no predictions found for this sample token')
    pred_boxes_raw = []
else:
    pred_boxes_raw = pred_data['results'][_sample_token]
    print(f'Loaded {len(pred_boxes_raw)} predicted boxes')
    from collections import Counter as _Counter
    print('Classes:', dict(_Counter(b['detection_name'] for b in pred_boxes_raw)))

# ── Transform predictions: global → ego frame ─────────────────────────────
pred_boxes = []
for b in pred_boxes_raw:
    center_g = np.array(b['translation'], dtype=np.float64)
    size     = np.array(b['size'],        dtype=np.float64)   # [W, L, H]
    rot_g    = Quaternion(b['rotation'])

    center_e = R_e2g.T @ (center_g - t_e2g)
    R_box_e  = R_e2g.T @ rot_g.rotation_matrix
    yaw      = float(np.arctan2(R_box_e[1, 0], R_box_e[0, 0]))

    half_w, half_l, half_h = size[0]/2, size[1]/2, size[2]/2
    _cx = np.array([ half_l,  half_l, -half_l, -half_l,  half_l,  half_l, -half_l, -half_l])
    _cy = np.array([ half_w, -half_w, -half_w,  half_w,  half_w, -half_w, -half_w,  half_w])
    _cz = np.array([-half_h, -half_h, -half_h, -half_h,  half_h,  half_h,  half_h,  half_h])
    corners_local = np.stack([_cx, _cy, _cz], axis=1)
    cos_y, sin_y  = np.cos(yaw), np.sin(yaw)
    R_yaw = np.array([[cos_y, -sin_y, 0], [sin_y, cos_y, 0], [0, 0, 1]])
    corners_ego   = (R_yaw @ corners_local.T).T + center_e

    pred_boxes.append({
        'center_e'   : center_e,
        'size'       : size,
        'yaw'        : yaw,
        'corners_ego': corners_ego,
        'corners_2d' : corners_ego[:4, :2],
        'category'   : b['detection_name'],
        'score'      : float(b['detection_score']),
    })

print(f'Transformed {len(pred_boxes)} predicted boxes to ego frame')

# ── OBB-aware free-space query ─────────────────────────────────────────────
# Fixes the AABB approximation used by query_box_freespace():
# For a rotated box the AABB over-counts — corner volumes outside the actual
# OBB are by definition empty, inflating free% even for perfectly correct boxes.
# This function iterates over AABB voxels but keeps only those whose centres
# fall inside the oriented box (local-frame extent check).
def query_box_freespace_obb(log_odds, center_e, size, yaw, grid_origin, voxel_size,
                            threshold_free=THRESHOLD_FREE, threshold_occ=THRESHOLD_OCC):
    half_w, half_l, half_h = size[0]/2, size[1]/2, size[2]/2
    gox, goy, goz = grid_origin
    vs = voxel_size
    NX_g, NY_g, NZ_g = log_odds.shape
    cx, cy, cz = center_e

    # Inverse rotation: ego → box-local frame (yaw only, Z unchanged)
    cos_y, sin_y = np.cos(yaw), np.sin(yaw)
    # R_ego2box.T = [[cos_y, sin_y], [-sin_y, cos_y]] in XY

    # AABB bounds for iteration (conservative: diagonal in XY + full Z)
    diag = float(np.sqrt(half_l**2 + half_w**2))
    ix_lo = max(0,     int((cx - diag       - gox) / vs) - 1)
    ix_hi = min(NX_g,  int((cx + diag       - gox) / vs) + 2)
    iy_lo = max(0,     int((cy - diag       - goy) / vs) - 1)
    iy_hi = min(NY_g,  int((cy + diag       - goy) / vs) + 2)
    iz_lo = max(0,     int((cz - half_h     - goz) / vs) - 1)
    iz_hi = min(NZ_g,  int((cz + half_h     - goz) / vs) + 2)

    if ix_lo >= ix_hi or iy_lo >= iy_hi or iz_lo >= iz_hi:
        return None

    # 2-D rotated-frame test (X/Y only, box-local forward/left) — kept 2-D so
    # its memory footprint doesn't multiply by the Z extent. A full 3-D
    # version here allocates hundreds of MB per box for large objects at
    # fine voxel size, since Z never affects this test.
    gx = np.float32(gox - cx) + (np.arange(ix_lo, ix_hi, dtype=np.float32) + 0.5) * np.float32(vs)
    gy = np.float32(goy - cy) + (np.arange(iy_lo, iy_hi, dtype=np.float32) + 0.5) * np.float32(vs)
    dx2, dy2 = np.meshgrid(gx, gy, indexing='ij')
    lx =  np.float32(cos_y) * dx2 + np.float32(sin_y) * dy2   # box-local X (forward)
    ly = -np.float32(sin_y) * dx2 + np.float32(cos_y) * dy2   # box-local Y (left)
    inside_xy = (np.abs(lx) <= half_l) & (np.abs(ly) <= half_w)
    if not inside_xy.any():
        return None

    # Z is independent of yaw — a plain 1-D range test.
    gz = goz + (np.arange(iz_lo, iz_hi) + 0.5) * vs
    inside_z = np.abs(gz - cz) <= half_h
    if not inside_z.any():
        return None

    lo_sub    = log_odds[ix_lo:ix_hi, iy_lo:iy_hi, iz_lo:iz_hi]   # view, not a copy
    lo_values = lo_sub[inside_xy][:, inside_z]   # both masks, no full-3-D temp
    n_total   = int(lo_values.size)
    if n_total == 0:
        return None
    n_free    = int((lo_values < threshold_free).sum())
    n_occ     = int((lo_values > threshold_occ).sum())
    n_unk     = int((lo_values == 0.0).sum())

    return {
        'n_total'   : n_total,
        'n_free'    : n_free,
        'n_occupied': n_occ,
        'n_unknown' : n_unk,
        'frac_free' : n_free / n_total,
        'frac_occ'  : n_occ  / n_total,
        'frac_unk'  : n_unk  / n_total,
        'lo_values' : lo_values,
    }

print('OBB query function defined.')


In [ ]:
# ── Experiment A: OBB voxel check against the free-space map ─────────────
#
# For each predicted box, project the OBB into the LiDAR voxel grid and
# classify every voxel it touches:
#
#   OCCUPIED → sensor hit here. Box correctly covers a surface.    (good)
#   UNKNOWN  → no beam reached here. Could be the object interior.  (neutral)
#   FREE     → sensor beam passed through and returned further away.
#              The sensor certified this space empty. The box            (BAD)
#              extending here is a hard geometric contradiction.
#
# The FREE fraction directly measures how much of the predicted box
# occupies space the sensor proved is empty — i.e. overshoot.

FREE_THRESHOLD = 0.10   # flag if >10% of OBB voxels are certified free

print('Experiment A — OBB voxels vs LiDAR free-space map\n')
hdr = (f'{"#":>3}  {"Category":<14}  {"Score":>5}  {"Range":>7}  '
       f'{"Free%":>6}  {"Occ%":>5}  {"Unk%":>5}  {"N_free":>7}  {"N_occ":>7}  {"Flag"}')
print(hdr)
print('\u2500' * 90)

pred_stats = []
n_flagged = 0
n_in_grid = 0

for i, pb in enumerate(pred_boxes):
    dist  = float(np.linalg.norm(pb['center_e'][:2]))
    stats = query_box_freespace_obb(
        log_odds, pb['center_e'], pb['size'], pb['yaw'], grid_origin, VOXEL_SIZE)

    if stats is None:
        pred_stats.append(None)
        print(f'{i:>3}  {pb["category"]:<14}  {pb["score"]:>5.3f}  {dist:>6.1f}m  (outside grid)')
        continue

    pred_stats.append(stats)
    n_in_grid += 1
    flag = '  *** OVERSHOOT' if stats['frac_free'] > FREE_THRESHOLD else ''
    if stats['frac_free'] > FREE_THRESHOLD:
        n_flagged += 1

    print(f'{i:>3}  {pb["category"]:<14}  {pb["score"]:>5.3f}  {dist:>6.1f}m  '
          f'{100*stats["frac_free"]:>5.1f}%  '
          f'{100*stats["frac_occ"]:>4.1f}%  '
          f'{100*stats["frac_unk"]:>4.1f}%  '
          f'{stats["n_free"]:>7,}  '
          f'{stats["n_occupied"]:>7,}  '
          f'{flag}')

print(f'\nSummary: {n_flagged}/{n_in_grid} in-grid predictions have '
      f'>{ 100*FREE_THRESHOLD:.0f}% of their OBB voxels in certified-free space')
if n_in_grid > 0:
    print(f'  = {n_flagged/n_in_grid*100:.1f}% of in-grid predictions show overshoot')



In [ ]:
# ── Experiment A BEV: predicted boxes coloured by OBB free% ──────────────
# Reuses bev_crop, ego_xy_to_px, r_lo/r_hi/c_lo/c_hi, orig_row/col from cell §7.
import matplotlib.patches as mpatches, matplotlib

cmap_pred = matplotlib.colormaps.get_cmap('YlOrRd')

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(bev_crop, origin='upper', interpolation='nearest')
ax.plot(orig_col, orig_row, 'w*', markersize=12, zorder=5)
ax.annotate('ego', (orig_col + 3, orig_row - 3), color='white', fontsize=8)

# GT boxes
for gb in gt_boxes:
    if float(np.linalg.norm(gb['center_e'][:2])) > BEV_MAX_RANGE_PLOT:
        continue
    draw_gt_box_bev(ax, gb['corners_2d'], gb['category'], c_lo, r_lo)

# Predicted boxes — colour = OBB free% (saturate at 40%)
for pb, st in zip(pred_boxes, pred_stats):
    dist = float(np.linalg.norm(pb['center_e'][:2]))
    if dist > BEV_MAX_RANGE_PLOT:
        continue
    if st is None:
        colour, lw = 'lightgrey', 1.0
    else:
        colour = cmap_pred(min(st['frac_free'] / 0.40, 1.0))
        lw     = 2.5 if st['frac_free'] > FREE_THRESHOLD else 1.5
    pts = np.array([ego_xy_to_px(c[0], c[1]) for c in pb['corners_2d']])
    pts[:, 0] -= c_lo;  pts[:, 1] -= r_lo
    poly = plt.Polygon(pts, fill=False, edgecolor=colour, linewidth=lw, zorder=4)
    ax.add_patch(poly)

# Axis ticks in metres
tick_m = np.arange(-40, 41, 10)
col_ticks = [orig_col - t / VOXEL_SIZE for t in tick_m]
row_ticks = [orig_row - t / VOXEL_SIZE for t in tick_m]
ax.set_xticks(col_ticks); ax.set_xticklabels([f'{t:+.0f}m' for t in tick_m])
ax.set_yticks(row_ticks); ax.set_yticklabels([f'{t:+.0f}m' for t in tick_m])
ax.set_xlim(0, c_hi - c_lo)
ax.set_ylim(r_hi - r_lo, 0)
ax.set_xlabel('\u2190 +Y (left)          -Y (right) \u2192')
ax.set_ylabel('+X (forward) \u2191              \u2193 -X (back)')
ax.set_title(f'{DATASET} \u2014 scene {SCENE_IDX}, frame {FRAME_IDX} | '
             f'Experiment A: OBB free% (flag > {100*FREE_THRESHOLD:.0f}%)')

handles = [
    mpatches.Patch(color=[100/255,149/255,237/255], label='Free (certified empty)'),
    mpatches.Patch(color=[220/255,50/255,50/255],   label='Occupied (LiDAR return)'),
    mpatches.Patch(color='yellow', label='GT box'),
    mpatches.Patch(color=cmap_pred(0.0),  label='Pred: 0% OBB free'),
    mpatches.Patch(color=cmap_pred(0.25), label='Pred: ~10% OBB free (threshold)'),
    mpatches.Patch(color=cmap_pred(1.0),  label='Pred: \u226540% OBB free (strong overshoot)'),
]
ax.legend(handles=handles, loc='upper right', fontsize=8)

sm = matplotlib.cm.ScalarMappable(cmap=cmap_pred, norm=plt.Normalize(vmin=0, vmax=40))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='OBB free-space voxels [%]', shrink=0.5, pad=0.02)

plt.tight_layout()
plt.show()


### 12.1 OBB free% query walkthrough — coarse candidates → refined membership

Illustrates the two-step voxel selection from §2.2 (`phase0_results.md`):

1. **Coarse candidate voxels** — an axis-aligned over-approximation (the diagonal-sized square
   `query_box_freespace_obb` uses) guaranteed to contain every voxel inside the rotated box, at the
   cost of also including voxels outside it.
2. **Refined membership** — each candidate voxel's centre is re-expressed along the box's own
   rotated (length, width) axes and kept only if `|lx| ≤ length/2` and `|ly| ≤ width/2`.

Picks one representative predicted box from Experiment A above (clear yaw, so the rotation is
visible) and plots both steps side by side, at the box's own Z-slice.


In [ ]:
# ── 12.1a. Pick a representative box + recompute the OBB query with intermediate masks ──
# Mirrors query_box_freespace_obb's geometry (same AABB + rotation-test logic) but keeps
# the boolean masks instead of collapsing them into counts, purely for plotting.

def obb_query_debug(log_odds, center_e, size, yaw, grid_origin, voxel_size):
    half_w, half_l, _ = size[0] / 2, size[1] / 2, size[2] / 2
    gox, goy, _ = grid_origin
    vs = voxel_size
    NX_g, NY_g, _ = log_odds.shape
    cx, cy, _ = center_e
    cos_y, sin_y = np.cos(yaw), np.sin(yaw)

    # Step 1 — coarse candidate region: axis-aligned square sized to the box's diagonal,
    # guaranteed to contain the rotated box at any yaw (same bound query_box_freespace_obb uses).
    diag = float(np.sqrt(half_l**2 + half_w**2))
    ix_lo = max(0,    int((cx - diag - gox) / vs) - 1)
    ix_hi = min(NX_g, int((cx + diag - gox) / vs) + 2)
    iy_lo = max(0,    int((cy - diag - goy) / vs) - 1)
    iy_hi = min(NY_g, int((cy + diag - goy) / vs) + 2)

    gx = gox + (np.arange(ix_lo, ix_hi) + 0.5) * vs
    gy = goy + (np.arange(iy_lo, iy_hi) + 0.5) * vs
    gx2, gy2 = np.meshgrid(gx, gy, indexing='ij')

    # Step 2 — refined membership: rotate each candidate's offset from the box centre into
    # the box's own (length, width) axes and test the true extent.
    dx, dy = gx2 - cx, gy2 - cy
    lx =  cos_y * dx + sin_y * dy   # along box length
    ly = -sin_y * dx + cos_y * dy   # along box width
    inside2d = (np.abs(lx) <= half_l) & (np.abs(ly) <= half_w)

    return dict(ix_lo=ix_lo, ix_hi=ix_hi, iy_lo=iy_lo, iy_hi=iy_hi, inside2d=inside2d)


# Representative box: prefer a *car* in Regime A (<15 m) with an off-axis yaw
# (rotation clearly visible) and the most overshoot, relaxing constraints in order
# (drop yaw requirement, then drop regime, then drop category) if nothing qualifies.
def _dist(pb):
    return float(np.linalg.norm(pb['center_e'][:2]))

_all = [(i, pb, st) for i, (pb, st) in enumerate(zip(pred_boxes, pred_stats))
        if st is not None and st['n_total'] >= 50]

_filters = [
    lambda pb: pb['category'] == 'car' and _dist(pb) < 15.0 and 15 <= abs(np.degrees(pb['yaw'])) % 90 <= 75,
    lambda pb: pb['category'] == 'car' and _dist(pb) < 15.0,
    lambda pb: pb['category'] == 'car',
    lambda pb: 15 <= abs(np.degrees(pb['yaw'])) % 90 <= 75,
    lambda pb: True,
]
_candidates = []
for f in _filters:
    _candidates = [(i, pb, st) for i, pb, st in _all if f(pb)]
    if _candidates:
        break
demo_i, demo_pb, demo_st = max(_candidates, key=lambda t: t[2]['frac_free'])

dbg = obb_query_debug(log_odds, demo_pb['center_e'], demo_pb['size'], demo_pb['yaw'],
                       grid_origin, VOXEL_SIZE)
n_candidate_slice = int(dbg['inside2d'].size)
n_inside_slice    = int(dbg['inside2d'].sum())

print(f'Demo box #{demo_i}: {demo_pb["category"]}, yaw={np.degrees(demo_pb["yaw"]):.1f}°, '
      f'3-D OBB free%={100*demo_st["frac_free"]:.1f}% (N={demo_st["n_total"]:,} voxels)')
print(f'This Z-slice: {n_candidate_slice} coarse candidates → {n_inside_slice} kept after the '
      f'rotation test ({100*n_inside_slice/n_candidate_slice:.1f}%)')


In [ ]:
# ── 12.1b. Two-panel figure: coarse candidates vs. refined membership ───────────
# Each voxel is drawn as its own bordered rectangle (not an alpha blend over the
# base image) so the selection is unambiguous regardless of what free/occupied/
# unknown colour sits underneath it.
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

margin_m = 3.0
half_l, half_w = demo_pb['size'][1] / 2, demo_pb['size'][0] / 2
diag = float(np.sqrt(half_l**2 + half_w**2))
cx, cy = demo_pb['center_e'][:2]

# Tight crop in pixel space around the demo box (1 voxel == 1 pixel in bev_display)
col_a, row_a = ego_xy_to_px(cx - diag - margin_m, cy - diag - margin_m)
col_b, row_b = ego_xy_to_px(cx + diag + margin_m, cy + diag + margin_m)
tc_lo, tc_hi = max(int(min(col_a, col_b)), 0), min(int(max(col_a, col_b)) + 1, NY)
tr_lo, tr_hi = max(int(min(row_a, row_b)), 0), min(int(max(row_a, row_b)) + 1, NX)
demo_crop = bev_display[tr_lo:tr_hi, tc_lo:tc_hi]

def _px(x, y):
    c, r = ego_xy_to_px(x, y)
    return c - tc_lo, r - tr_lo

def _voxel_rect_px(ix, iy):
    # World-space edges of voxel (ix, iy) -> a pixel-space rectangle. The BEV
    # pixel grid is an axis-preserving flip of the voxel grid, so a voxel's
    # world-aligned edges always map back to an axis-aligned pixel rectangle.
    x0, y0 = grid_origin[0] + ix * VOXEL_SIZE, grid_origin[1] + iy * VOXEL_SIZE
    x1, y1 = x0 + VOXEL_SIZE,                  y0 + VOXEL_SIZE
    p0, p1 = _px(x0, y0), _px(x1, y1)
    x_lo, x_hi = sorted([p0[0], p1[0]])
    y_lo, y_hi = sorted([p0[1], p1[1]])
    return x_lo, y_lo, x_hi - x_lo, y_hi - y_lo

fig, axes = plt.subplots(1, 2, figsize=(14, 7.5))
titles = [
    f'Step 1 — coarse candidates (AABB over-approx.)\n{n_candidate_slice} voxels in this Z-slice',
    f'Step 2 — refined membership (rotated-frame test)\n'
    f'{n_inside_slice}/{n_candidate_slice} kept ({100*n_inside_slice/n_candidate_slice:.1f}%)',
]
for ax, title in zip(axes, titles):
    ax.imshow(demo_crop, origin='upper', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

# Panel 1 — every coarse candidate voxel, outlined + lightly filled
for ix in range(dbg['ix_lo'], dbg['ix_hi']):
    for iy in range(dbg['iy_lo'], dbg['iy_hi']):
        x, y, w, h = _voxel_rect_px(ix, iy)
        axes[0].add_patch(mpatches.Rectangle((x, y), w, h, facecolor='orange', alpha=0.35,
                                              edgecolor='darkorange', linewidth=0.7, zorder=4))

# Panel 2 — kept voxels solid green; rejected candidates shown as a thin dotted
# outline only, so it's visible they were considered and then excluded.
for ix in range(dbg['ix_lo'], dbg['ix_hi']):
    for iy in range(dbg['iy_lo'], dbg['iy_hi']):
        x, y, w, h = _voxel_rect_px(ix, iy)
        kept = dbg['inside2d'][ix - dbg['ix_lo'], iy - dbg['iy_lo']]
        if kept:
            axes[1].add_patch(mpatches.Rectangle((x, y), w, h, facecolor='limegreen', alpha=0.50,
                                                  edgecolor='darkgreen', linewidth=0.7, zorder=4))
        else:
            axes[1].add_patch(mpatches.Rectangle((x, y), w, h, fill=False,
                                                  edgecolor='darkorange', linestyle=':',
                                                  linewidth=0.6, alpha=0.8, zorder=3))

# Box outline + centre, both panels
for ax in axes:
    pts = np.array([_px(c[0], c[1]) for c in demo_pb['corners_2d']])
    ax.add_patch(plt.Polygon(pts, closed=True, fill=False, edgecolor='white', linewidth=2.2, zorder=6))
    ax.plot(*_px(cx, cy), 'w+', markersize=10, mew=2, zorder=7)

# Panel 2 only: rotated length/width axes. Labels are offset in fixed *display
# points* (not data units) so the spacing looks right regardless of how small
# the object/crop is — a data-space offset like "8 voxels" is huge for a
# bicycle and tiny for a truck, which is what made labels fly off earlier.
cos_y, sin_y = np.cos(demo_pb['yaw']), np.sin(demo_pb['yaw'])
center_px = np.array(_px(cx, cy))
fwd_px  = np.array(_px(cx + cos_y * half_l,  cy + sin_y * half_l))
side_px = np.array(_px(cx - sin_y * half_w,  cy + cos_y * half_w))
axes[1].annotate('', xy=fwd_px,  xytext=center_px, arrowprops=dict(arrowstyle='->', color='cyan',    linewidth=2), zorder=7)
axes[1].annotate('', xy=side_px, xytext=center_px, arrowprops=dict(arrowstyle='->', color='magenta', linewidth=2), zorder=7)
axes[1].annotate('length axis (lx)', xy=fwd_px, xycoords='data', xytext=(8, -4),
                  textcoords='offset points', color='cyan', fontsize=8, va='center', zorder=7)
axes[1].annotate('width axis (ly)', xy=side_px, xycoords='data', xytext=(8, 4),
                  textcoords='offset points', color='magenta', fontsize=8, va='center', zorder=7)

for ax in axes:
    ax.set_xlim(0, demo_crop.shape[1])
    ax.set_ylim(demo_crop.shape[0], 0)

fig.suptitle(f'{DATASET} — box #{demo_i} ({demo_pb["category"]}, yaw={np.degrees(demo_pb["yaw"]):.1f}°) '
             f'— OBB free% query walkthrough', fontsize=12, y=0.99)

legend_handles = [
    mpatches.Patch(color=[100/255, 149/255, 237/255], label='Free (certified empty)'),
    mpatches.Patch(color=[220/255,  50/255,  50/255], label='Occupied (LiDAR return)'),
    mpatches.Patch(color=[200/255, 200/255, 200/255], label='Unknown'),
    mpatches.Patch(facecolor='orange', alpha=0.5, edgecolor='darkorange', label='Candidate voxel (Step 1)'),
    mpatches.Patch(facecolor='limegreen', alpha=0.6, edgecolor='darkgreen', label='Kept — inside rotated box (Step 2)'),
    mpatches.Patch(fill=False, edgecolor='darkorange', linestyle=':', label='Rejected candidate (Step 2)'),
    Line2D([0], [0], color='white',   linewidth=2.2, label='Predicted OBB (true footprint)'),
    Line2D([0], [0], color='cyan',    linewidth=2,   label='Length axis (lx)'),
    Line2D([0], [0], color='magenta', linewidth=2,   label='Width axis (ly)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, fontsize=8,
           bbox_to_anchor=(0.5, -0.02), frameon=True)

fig.tight_layout(rect=[0, 0.12, 1, 0.90])

out_path = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'phase0_obb_query_walkthrough_{DATASET}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()


### 12.2 Regime comparison — GT vs Predicted OBB across A/B/C

For the scene/frame selected in §0, finds one matched GT+prediction pair per range regime
(A < 15m, B 15-30m, C > 30m) — preferring a single category present in all three regimes, so the
comparison is apples-to-apples — and plots the GT box, the predicted box, and the underlying voxels
side by side, annotated with:

- **floor%** — the sensor-geometry reference free% for that (dataset, category, 5m range bin), from
  `experiment_a_gt_baseline_{DATASET}.csv`, using the exact binning §15 uses.
- **pred_free%** — this specific predicted box's own measured OBB free% (§12).
- **excess** — `pred_free% − floor%`, the corrected overshoot signal (§15.2).

If the current frame doesn't have a matched pair in all three regimes, change `SCENE_IDX` /
`FRAME_IDX` in §0 and re-run from §2 onward.


In [ ]:
# ── 12.2a. Find a matched GT+prediction pair per regime ─────────────────────
# Reuses the exact category mapping and range-binning from §15 (Phase 0.5) so
# the "floor%" shown here matches the aggregate GT baseline exactly.
import pandas as pd

NUSCENES_GT_TO_PRED = {
    'vehicle.car':                          'car',
    'vehicle.truck':                        'truck',
    'vehicle.bus.bendy':                    'bus',
    'vehicle.bus.rigid':                    'bus',
    'vehicle.trailer':                      'trailer',
    'vehicle.motorcycle':                   'motorcycle',
    'vehicle.bicycle':                      'bicycle',
    'vehicle.construction':                 'construction_vehicle',
    'human.pedestrian.adult':               'pedestrian',
    'human.pedestrian.child':               'pedestrian',
    'human.pedestrian.construction_worker': 'pedestrian',
    'human.pedestrian.personal_mobility':   'pedestrian',
    'human.pedestrian.police_officer':      'pedestrian',
}

def gt_category_to_pred(cat):
    return NUSCENES_GT_TO_PRED.get(cat, cat)   # ECP GT already uses prediction names

def regime_of(dist):
    return 'A' if dist < 15.0 else ('B' if dist < 30.0 else 'C')

def best_match(regime, category=None):
    best = None
    for gb in gt_boxes:
        gcat  = gt_category_to_pred(gb['category'])
        gdist = float(np.linalg.norm(gb['center_e'][:2]))
        if regime_of(gdist) != regime:
            continue
        if category is not None and gcat != category:
            continue
        for pidx, pb in enumerate(pred_boxes):
            if pb['category'] != gcat:
                continue
            pdist = float(np.linalg.norm(pb['center_e'][:2]))
            if regime_of(pdist) != regime:
                continue
            d = abs(gdist - pdist)
            if best is None or d < best[0]:
                best = (d, gb, pb, pidx, gcat, gdist, pdist)
    return best

# Prefer one category with a matched pair in all three regimes (apples-to-apples);
# fall back to the best available match per regime otherwise.
candidate_categories = sorted(
    {gt_category_to_pred(gb['category']) for gb in gt_boxes} &
    {pb['category'] for pb in pred_boxes}
)
chosen_category = next(
    (c for c in candidate_categories if all(best_match(r, c) for r in ['A', 'B', 'C'])),
    None
)

selections = {r: (best_match(r, chosen_category) if chosen_category else best_match(r))
              for r in ['A', 'B', 'C']}

print(f'Chosen category for cross-regime comparison: {chosen_category or "(mixed — best available per regime)"}\n')
for r in ['A', 'B', 'C']:
    m = selections[r]
    if m is None:
        print(f'Regime {r}: no matched GT+prediction pair in this frame.')
    else:
        d, gb, pb, pidx, cat, gdist, pdist = m
        print(f'Regime {r}: {cat:10s}  GT range={gdist:5.2f}m  pred range={pdist:5.2f}m  (match Δ={d:.2f}m)')

if any(v is None for v in selections.values()):
    print('\nNOT all three regimes have a matched example in this frame.')
    print('Change SCENE_IDX / FRAME_IDX in §0 and re-run from §2 onward to find one that does.')

# ── Sensor-geometry floor%, same (dataset, category, range_bin) lookup as §15 ──
gt_baseline_df = pd.read_csv(DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'experiment_a_gt_baseline_{DATASET}.csv')
gt_baseline_df['category_pred'] = gt_baseline_df['category'].apply(
    lambda c: c if DATASET == 'ecp' else NUSCENES_GT_TO_PRED.get(c))
gt_baseline_df = gt_baseline_df[gt_baseline_df['category_pred'].notna()].copy()
gt_baseline_df['range_bin'] = (gt_baseline_df['range_m'] // 5.0) * 5.0

def lookup_floor(category, dist):
    rb  = (dist // 5.0) * 5.0
    sub = gt_baseline_df[(gt_baseline_df.category_pred == category) & (gt_baseline_df.range_bin == rb)]
    return float(sub.frac_free.median()) if len(sub) else None


In [ ]:
# ── 12.2b. Three-panel figure: GT vs predicted OBB, one per regime ──────────
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

regime_labels = {'A': 'Regime A (< 15 m)', 'B': 'Regime B (15–30 m)', 'C': 'Regime C (> 30 m)'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))

for ax, r in zip(axes, ['A', 'B', 'C']):
    m = selections[r]
    if m is None:
        ax.set_title(f'{regime_labels[r]}\n(no matched example in this frame)', fontsize=10)
        ax.axis('off')
        continue

    d, gb, pb, pidx, cat, gdist, pdist = m
    pred_free = pred_stats[pidx]['frac_free'] if pred_stats[pidx] is not None else None
    floor     = lookup_floor(cat, pdist)

    cx = (gb['center_e'][0] + pb['center_e'][0]) / 2
    cy = (gb['center_e'][1] + pb['center_e'][1]) / 2
    diag_g = float(np.sqrt((gb['size'][1]/2)**2 + (gb['size'][0]/2)**2))
    diag_p = float(np.sqrt((pb['size'][1]/2)**2 + (pb['size'][0]/2)**2))
    sep    = max(abs(gb['center_e'][0]-pb['center_e'][0]), abs(gb['center_e'][1]-pb['center_e'][1]))
    half_span = max(diag_g, diag_p) + sep/2 + 2.0   # +2m context margin

    col_a, row_a = ego_xy_to_px(cx - half_span, cy - half_span)
    col_b, row_b = ego_xy_to_px(cx + half_span, cy + half_span)
    tc_lo, tc_hi = max(int(min(col_a, col_b)), 0), min(int(max(col_a, col_b)) + 1, NY)
    tr_lo, tr_hi = max(int(min(row_a, row_b)), 0), min(int(max(row_a, row_b)) + 1, NX)
    crop = bev_display[tr_lo:tr_hi, tc_lo:tc_hi]

    def _px2(x, y, tc_lo=tc_lo, tr_lo=tr_lo):
        c, rr = ego_xy_to_px(x, y)
        return c - tc_lo, rr - tr_lo

    ax.imshow(crop, origin='upper', interpolation='nearest')
    gpts = np.array([_px2(c[0], c[1]) for c in gb['corners_2d']])
    ppts = np.array([_px2(c[0], c[1]) for c in pb['corners_2d']])
    ax.add_patch(plt.Polygon(gpts, closed=True, fill=False, edgecolor='yellow', linewidth=2.2, zorder=5))
    ax.add_patch(plt.Polygon(ppts, closed=True, fill=False, edgecolor='cyan',   linewidth=2.2,
                              linestyle='--', zorder=6))
    ax.set_xlim(0, crop.shape[1]); ax.set_ylim(crop.shape[0], 0)
    ax.set_xticks([]); ax.set_yticks([])

    free_str   = f'{100*pred_free:.1f}%' if pred_free is not None else 'n/a'
    floor_str  = f'{100*floor:.1f}%' if floor is not None else 'n/a'
    excess_str = (f'{100*(pred_free-floor):+.1f}pp'
                  if (pred_free is not None and floor is not None) else 'n/a')
    ax.set_title(f'{regime_labels[r]} — {cat}\n'
                 f'floor%={floor_str}   pred_free%={free_str}   excess={excess_str}', fontsize=10)

legend_handles = [
    mpatches.Patch(color=[100/255, 149/255, 237/255], label='Free (certified empty)'),
    mpatches.Patch(color=[220/255,  50/255,  50/255], label='Occupied (LiDAR return)'),
    mpatches.Patch(color=[200/255, 200/255, 200/255], label='Unknown'),
    Line2D([0], [0], color='yellow', linewidth=2.2, label='GT box'),
    Line2D([0], [0], color='cyan',   linewidth=2.2, linestyle='--', label='Predicted box'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.04))
fig.suptitle(f'{DATASET} — scene {SCENE_IDX}, frame {FRAME_IDX} — '
             f'GT vs predicted OBB across range regimes (voxel={VOXEL_SIZE}m)', fontsize=13, y=1.03)
fig.tight_layout(rect=[0, 0.08, 1, 0.93])

out_path = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'phase0_regime_comparison_{DATASET}.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()


### 12.3 Regime comparison — all categories, both datasets

§12.2 showed one category (whichever had a matched GT+prediction pair in all three regimes) in
the single demo frame. This section instead searches **every annotated frame** in the current
`DATASET` for the best matched pair per (category, regime), and renders one row per category.

**How it stays cheap:** most categories don't have a usable example in the single demo frame, so a
naive search would need to load LiDAR + build a voxel grid for many frames. Instead, §12.3a first
does a **greedy set-cover search using only the summary CSVs** (fast, no LiDAR/NuScenes needed) to
find the *smallest* set of frames that together cover every available (category, regime) slot —
typically a handful of frames, not dozens. Only those frames get their voxel grid rebuilt.

**Memory safety — read before running §12.3b:** it reuses this kernel's already-loaded `nusc` and
`pred_data` for the *current* `DATASET` (no second NuScenes load). To get figures for **both**
datasets: run §12.3a/b once with `DATASET = 'ecp'`, then **restart the kernel**, set
`DATASET = 'nuscenes_mini'` in §0, and re-run from the top through §12.3b again. Changing `DATASET`
and re-running without restarting would load a second NuScenes index in the same process — that
combination (two NuScenes indexes alive at once) is what caused this notebook's OOM crashes earlier.
§12.3c (plotting) is cheap pandas/matplotlib only and safely picks up whichever dataset(s) have
already been gathered.


In [ ]:
# ── 12.3a. Greedy frame selection (pandas only — no LiDAR/NuScenes) ─────────
# For every (category, regime) that has both a prediction and a GT example
# somewhere in this dataset, find the smallest set of frames that together
# provide a matched pair for as many of those slots as possible.
import pandas as pd

NUSCENES_GT_TO_PRED = {
    'vehicle.car':                          'car',
    'vehicle.truck':                        'truck',
    'vehicle.bus.bendy':                    'bus',
    'vehicle.bus.rigid':                    'bus',
    'vehicle.trailer':                      'trailer',
    'vehicle.motorcycle':                   'motorcycle',
    'vehicle.bicycle':                      'bicycle',
    'vehicle.construction':                 'construction_vehicle',
    'human.pedestrian.adult':               'pedestrian',
    'human.pedestrian.child':               'pedestrian',
    'human.pedestrian.construction_worker': 'pedestrian',
    'human.pedestrian.personal_mobility':   'pedestrian',
    'human.pedestrian.police_officer':      'pedestrian',
}

def gt_cat_to_pred(c):
    return NUSCENES_GT_TO_PRED.get(c, c)   # ECP GT already uses prediction names

def regime_of_dist(dist):
    return 'A' if dist < 15.0 else ('B' if dist < 30.0 else 'C')

def select_regime_examples(dataset):
    pred_df = pd.read_csv(DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'experiment_a_{dataset}.csv')
    gt_df   = pd.read_csv(DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' /
                           f'experiment_a_gt_baseline_{dataset}.csv').copy()
    gt_df['category_pred'] = gt_df['category'].apply(gt_cat_to_pred)
    gt_df = gt_df[gt_df['category_pred'].notna()]
    gt_df['range_bin'] = (gt_df['range_m'] // 5.0) * 5.0

    categories = sorted(set(pred_df.category.unique()) & set(gt_df.category_pred.unique()))
    needed = set()
    for cat in categories:
        for r in ['A', 'B', 'C']:
            if ((pred_df.category == cat) & (pred_df.regime == r)).any() and \
               ((gt_df.category_pred == cat) & (gt_df.regime == r)).any():
                needed.add((cat, r))

    pred_keys = pred_df.groupby('sample_token').apply(
        lambda g: set(zip(g.category, g.regime)), include_groups=False).to_dict()
    gt_keys = gt_df.groupby('sample_token').apply(
        lambda g: set(zip(g.category_pred, g.regime)), include_groups=False).to_dict()
    tokens = set(pred_keys) & set(gt_keys)
    cov = {tok: (pred_keys[tok] & gt_keys[tok]) & needed for tok in tokens}
    cov = {tok: s for tok, s in cov.items() if s}

    remaining = set(needed)
    chosen_tokens, slot_assignment = [], {}
    while remaining:
        best_tok, best_gain = None, 0
        for tok, slots in cov.items():
            gain = len(slots & remaining)
            if gain > best_gain:
                best_tok, best_gain = tok, gain
        if best_tok is None:
            break
        chosen_tokens.append(best_tok)
        won = cov[best_tok] & remaining
        for s in won:
            slot_assignment[s] = best_tok
        remaining -= won

    print(f'{dataset}: {len(categories)} categories, {len(needed)} possible slots, '
          f'{len(remaining)} uncovered, {len(chosen_tokens)} frames needed')
    if remaining:
        print('  uncovered (no frame has both a GT and a prediction match):', sorted(remaining))

    return dict(categories=categories, slot_assignment=slot_assignment,
                chosen_tokens=chosen_tokens, gt_df=gt_df, uncovered=remaining)

_selection = select_regime_examples(DATASET)


In [ ]:
# ── 12.3b. Gather crops for the selected frames (reuses this kernel's nusc/pred_data) ──
# Only re-runs ray casting for the handful of frames chosen in §12.3a — the
# main log_odds array is reused (zeroed + rebuilt per frame), not reallocated,
# to keep memory bounded regardless of how many frames are visited.
import sys, pickle
from collections import defaultdict
sys.path.insert(0, str(DEV_ROOT / 'contribution_ideas' / 'phase0' / 'scripts'))
import experiment_a_batch as ea_batch   # raytrace + query_box_freespace_obb, for exact
                                         # consistency with the reported statistics

def _corners_2d(center_e, size, yaw):
    half_w, half_l = size[0]/2, size[1]/2
    cx_ = np.array([half_l, half_l, -half_l, -half_l])
    cy_ = np.array([half_w, -half_w, -half_w, half_w])
    cos_y, sin_y = np.cos(yaw), np.sin(yaw)
    R2 = np.array([[cos_y, -sin_y], [sin_y, cos_y]])
    return (R2 @ np.stack([cx_, cy_], axis=1).T).T + center_e[:2]

def gather_regime_examples(dataset, selection):
    tok_slots = defaultdict(list)
    for slot, tok in selection['slot_assignment'].items():
        tok_slots[tok].append(slot)

    results = {}
    for tok in selection['chosen_tokens']:
        slots_here = tok_slots[tok]
        print(f'Frame {tok}: {slots_here}')
        sample  = nusc.get('sample', tok)
        lid_sd  = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
        ep      = nusc.get('ego_pose', lid_sd['ego_pose_token'])
        cs      = nusc.get('calibrated_sensor', lid_sd['calibrated_sensor_token'])
        t_e2g_  = np.array(ep['translation']); R_e2g_ = Quaternion(ep['rotation']).rotation_matrix
        t_l2e_  = np.array(cs['translation']); R_l2e_ = Quaternion(cs['rotation']).rotation_matrix

        lidar_path = nusc.get_sample_data_path(sample['data']['LIDAR_TOP'])
        stride = 5   # both datasets are (x,y,z,intensity,ring) — a previous 'ecp:5 else 4'
                     # here silently scrambled every nuScenes point; see §2.1 note.
        raw = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, stride)[:, :3]
        pts_ego_ = (R_l2e_ @ raw.T).T + t_l2e_
        so_ = t_l2e_.astype(np.float32)
        rngs = np.linalg.norm(pts_ego_ - so_, axis=1)
        m = (rngs >= RAY_MIN_RANGE) & (rngs <= RAY_MAX_RANGE)
        pts_cast_ = pts_ego_[m].astype(np.float32)

        log_odds[:] = 0.0
        ea_batch.raytrace(log_odds, pts_cast_, so_, grid_origin, VOXEL_SIZE,
                           LO_FREE, LO_OCC, LO_MIN, LO_MAX)

        iz_center = int((BEV_Z_SLICE_HEIGHT - GRID_Z_MIN) / VOXEL_SIZE)
        iz_half   = max(1, int(BEV_Z_HALF_WIDTH / VOXEL_SIZE))
        iz_lo, iz_hi = max(0, iz_center-iz_half), min(NZ, iz_center+iz_half+1)
        # NOTE (fixed 2026-08-31): a plain .max(axis=2) over signed log-odds is wrong here --
        # untouched voxels are exactly 0, free voxels are negative, and 0 > any negative value,
        # so a single untouched voxel anywhere in the Z-band masks real free-space evidence from
        # every other layer in that column. Use an any()-based, occupied-takes-priority reduction
        # instead (matches how the actual OBB queries treat 3D evidence).
        _occ_mask  = (log_odds[:, :, iz_lo:iz_hi] > THRESHOLD_OCC).any(axis=2)
        _free_mask = (log_odds[:, :, iz_lo:iz_hi] < THRESHOLD_FREE).any(axis=2) & ~_occ_mask
        bev_rgb = np.full((NX, NY, 3), 200, dtype=np.uint8)
        bev_rgb[_free_mask] = [100, 149, 237]
        bev_rgb[_occ_mask]  = [220, 50, 50]
        bev_disp = bev_rgb[::-1, ::-1, :]

        gt_here, pred_here = [], []
        for ann_token in sample['anns']:
            ann = nusc.get('sample_annotation', ann_token)
            cat = gt_cat_to_pred(ann['category_name'])
            if cat is None:
                continue
            center_e = R_e2g_.T @ (np.array(ann['translation']) - t_e2g_)
            R_box_e  = R_e2g_.T @ Quaternion(ann['rotation']).rotation_matrix
            yaw  = float(np.arctan2(R_box_e[1,0], R_box_e[0,0]))
            size = np.array(ann['size'])
            dist = float(np.linalg.norm(center_e[:2]))
            gt_here.append(dict(category=cat, center_e=center_e, size=size, yaw=yaw,
                                 dist=dist, regime=regime_of_dist(dist),
                                 corners_2d=_corners_2d(center_e, size, yaw)))
        for b in pred_data['results'].get(tok, []):
            center_e = R_e2g_.T @ (np.array(b['translation']) - t_e2g_)
            R_box_e  = R_e2g_.T @ Quaternion(b['rotation']).rotation_matrix
            yaw  = float(np.arctan2(R_box_e[1,0], R_box_e[0,0]))
            size = np.array(b['size'])
            dist = float(np.linalg.norm(center_e[:2]))
            pred_here.append(dict(category=b['detection_name'], center_e=center_e, size=size, yaw=yaw,
                                   dist=dist, regime=regime_of_dist(dist),
                                   corners_2d=_corners_2d(center_e, size, yaw)))

        def _to_px(x, y):
            row = NX - 1 - (x - GRID_X_MIN) / VOXEL_SIZE
            col = NY - 1 - (y - GRID_Y_MIN) / VOXEL_SIZE
            return float(col), float(row)

        for (cat, regime) in slots_here:
            gcands = [g for g in gt_here if g['category']==cat and g['regime']==regime]
            pcands = [p for p in pred_here if p['category']==cat and p['regime']==regime]
            if not gcands or not pcands:
                print(f'  WARNING: {(cat, regime)} not found in {tok} on re-check — skipped')
                continue
            _, gb, pb = min(((abs(g['dist']-p['dist']), g, p) for g in gcands for p in pcands),
                             key=lambda t: t[0])

            stat = ea_batch.query_box_freespace_obb(log_odds, pb['center_e'], pb['size'], pb['yaw'])
            pred_free = stat['frac_free'] if stat else None
            rb = (pb['dist'] // 5.0) * 5.0
            sub = selection['gt_df'][(selection['gt_df'].category_pred==cat) &
                                      (selection['gt_df'].range_bin==rb)]
            floor = float(sub.frac_free.median()) if len(sub) else None

            cx = (gb['center_e'][0]+pb['center_e'][0])/2
            cy = (gb['center_e'][1]+pb['center_e'][1])/2
            diag_g = float(np.sqrt((gb['size'][1]/2)**2+(gb['size'][0]/2)**2))
            diag_p = float(np.sqrt((pb['size'][1]/2)**2+(pb['size'][0]/2)**2))
            sep = max(abs(gb['center_e'][0]-pb['center_e'][0]), abs(gb['center_e'][1]-pb['center_e'][1]))
            half_span = max(diag_g, diag_p) + sep/2 + 2.0

            col_a, row_a = _to_px(cx-half_span, cy-half_span)
            col_b, row_b = _to_px(cx+half_span, cy+half_span)
            tc_lo, tc_hi = max(int(min(col_a,col_b)),0), min(int(max(col_a,col_b))+1, NY)
            tr_lo, tr_hi = max(int(min(row_a,row_b)),0), min(int(max(row_a,row_b))+1, NX)
            crop = bev_disp[tr_lo:tr_hi, tc_lo:tc_hi].copy()

            gpts = np.array([[c[0]-tc_lo, c[1]-tr_lo] for c in
                              [_to_px(*p) for p in gb['corners_2d']]])
            ppts = np.array([[c[0]-tc_lo, c[1]-tr_lo] for c in
                              [_to_px(*p) for p in pb['corners_2d']]])

            results[(cat, regime)] = dict(crop=crop, gpts=gpts, ppts=ppts,
                                           pred_free=pred_free, floor=floor, sample_token=tok)
    return results

_gathered = gather_regime_examples(DATASET, _selection)

out_pkl = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'regime_examples_{DATASET}.pkl'
with open(out_pkl, 'wb') as f:
    pickle.dump(dict(categories=_selection['categories'], results=_gathered,
                      uncovered=_selection['uncovered']), f)
print(f'Saved {len(_gathered)} examples -> {out_pkl}')


In [ ]:
# ── 12.3c. Plot — safe to (re)run any time, no LiDAR/NuScenes needed ────────
# Renders one figure per dataset that has a saved regime_examples_{dataset}.pkl,
# whether it was gathered in this kernel session or an earlier one.
import pickle

def plot_regime_examples(dataset):
    pkl_path = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'regime_examples_{dataset}.pkl'
    if not pkl_path.exists():
        print(f'{dataset}: no {pkl_path.name} yet — run §12.3a/b with DATASET={dataset!r} first')
        return
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    results = data['results']
    all_categories = data['categories']
    categories = [c for c in all_categories if any((c, r) in results for r in ['A', 'B', 'C'])]
    skipped = [c for c in all_categories if c not in categories]
    if skipped:
        print(f'{dataset}: no example in any regime for {skipped} — omitted from the figure')
    if not categories:
        print(f'{dataset}: nothing to plot')
        return

    n_cat = len(categories)
    fig, axes = plt.subplots(n_cat, 3, figsize=(15, 4.6 * n_cat), squeeze=False)
    regime_labels = {'A': 'Regime A (< 15 m)', 'B': 'Regime B (15-30 m)', 'C': 'Regime C (> 30 m)'}

    for row, cat in enumerate(categories):
        for col, r in enumerate(['A', 'B', 'C']):
            ax = axes[row][col]
            ex = results.get((cat, r))
            if ex is None:
                ax.set_title(f'{cat} — {regime_labels[r]}\n(no matched example available)', fontsize=9)
                ax.axis('off')
                continue
            ax.imshow(ex['crop'], origin='upper', interpolation='nearest')
            ax.add_patch(plt.Polygon(ex['gpts'], closed=True, fill=False, edgecolor='yellow',
                                      linewidth=2.0, zorder=5))
            ax.add_patch(plt.Polygon(ex['ppts'], closed=True, fill=False, edgecolor='cyan',
                                      linewidth=2.0, linestyle='--', zorder=6))
            ax.set_xlim(0, ex['crop'].shape[1]); ax.set_ylim(ex['crop'].shape[0], 0)
            ax.set_xticks([]); ax.set_yticks([])
            pf, fl = ex['pred_free'], ex['floor']
            free_str  = f'{100*pf:.1f}%' if pf is not None else 'n/a'
            floor_str = f'{100*fl:.1f}%' if fl is not None else 'n/a'
            excess_str = f'{100*(pf-fl):+.1f}pp' if (pf is not None and fl is not None) else 'n/a'
            ax.set_title(f'{cat} — {regime_labels[r]}\n'
                         f'floor%={floor_str}  pred_free%={free_str}  excess={excess_str}', fontsize=9)

    fig.tight_layout()
    legend_handles = [
        mpatches.Patch(color=[100/255, 149/255, 237/255], label='Free (certified empty)'),
        mpatches.Patch(color=[220/255,  50/255,  50/255], label='Occupied (LiDAR return)'),
        mpatches.Patch(color=[200/255, 200/255, 200/255], label='Unknown'),
        Line2D([0], [0], color='yellow', linewidth=2.0, label='GT box'),
        Line2D([0], [0], color='cyan',   linewidth=2.0, linestyle='--', label='Predicted box'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', ncol=5, fontsize=10, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f'{dataset} — GT vs predicted OBB across range regimes, all categories '
                 f'(voxel={VOXEL_SIZE}m)', fontsize=14, y=1.0)

    out_path = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / f'phase0_regime_comparison_all_categories_{dataset}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'Saved: {out_path}  ({n_cat} categories x 3 regimes)')
    plt.show()

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
for _ds in ['ecp', 'nuscenes_mini']:
    plot_regime_examples(_ds)


## 13. Phase 0.4 — Overshoot distribution per class and range regime

**Goal (§28 Phase 0, item 0.4):** Plot OBB free% as a function of range for every predicted box, stratified by class and by the §3 regime proxy:

- **Regime A proxy:** range < 15 m — dense LiDAR, free-space signal strongest
- **Regime B proxy:** 15–30 m — moderate LiDAR
- **Regime C proxy:** > 30 m — sparse/no LiDAR, signal collapses

This is the go/no-go figure. If overshoot concentrates in Regime A and collapses in Regime C, the regime structure is validated and Phase 1 is justified.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Collect per-prediction records ────────────────────────────────────────
records = []
for pb, st in zip(pred_boxes, pred_stats):
    if st is None:
        continue
    dist = float(np.linalg.norm(pb['center_e'][:2]))
    records.append({
        'range'   : dist,
        'frac_free': st['frac_free'],
        'category': pb['category'],
        'flagged' : st['frac_free'] > FREE_THRESHOLD,
    })

# Regime boundaries (range proxy)
REGIME_A_MAX = 15.0
REGIME_B_MAX = 30.0

def regime(d):
    if d < REGIME_A_MAX:  return 'A (<15 m)'
    if d < REGIME_B_MAX:  return 'B (15-30 m)'
    return 'C (>30 m)'

CLASS_COLORS = {
    'car'       : '#2196F3',
    'truck'     : '#9C27B0',
    'trailer'   : '#795548',
    'bus'       : '#FF5722',
    'pedestrian': '#4CAF50',
    'bicycle'   : '#FF9800',
    'motorcycle': '#F44336',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'{DATASET} — scene {SCENE_IDX}, frame {FRAME_IDX} | '
    'Phase 0.4: OBB free% per class and regime',
    fontsize=12)

# ── Left: scatter — range vs OBB free% ────────────────────────────────────
ax = axes[0]
ax.axvspan(0,            REGIME_A_MAX, alpha=0.08, color='red',   label='_A')
ax.axvspan(REGIME_A_MAX, REGIME_B_MAX, alpha=0.08, color='orange',label='_B')
ax.axvspan(REGIME_B_MAX, 55,           alpha=0.08, color='grey',  label='_C')
ax.axhline(FREE_THRESHOLD * 100, color='black', lw=1.2,
           ls='--', label=f'Threshold ({100*FREE_THRESHOLD:.0f}%)')

seen_classes = set()
for r in records:
    cat   = r['category']
    color = CLASS_COLORS.get(cat, 'grey')
    label = cat if cat not in seen_classes else '_'
    seen_classes.add(cat)
    marker = 'D' if r['flagged'] else 'o'
    ax.scatter(r['range'], r['frac_free'] * 100,
               color=color, marker=marker,
               s=60, edgecolors='k', linewidths=0.4,
               label=label, zorder=3)

ax.set_xlabel('Range to ego [m]')
ax.set_ylabel('OBB free-space voxels [%]')
ax.set_xlim(0, 55)
ax.set_ylim(-2, 100)
ax.set_title('Scatter: range vs overshoot')

# Regime labels
ax.text(7,   95, 'Regime A', ha='center', fontsize=8, color='darkred')
ax.text(22,  95, 'Regime B', ha='center', fontsize=8, color='darkorange')
ax.text(42,  95, 'Regime C', ha='center', fontsize=8, color='grey')

# Legend: classes + threshold
class_handles = [mpatches.Patch(color=CLASS_COLORS.get(c,'grey'), label=c)
                 for c in sorted(seen_classes)]
thr_handle = plt.Line2D([0],[0], color='black', ls='--',
                         label=f'Threshold ({100*FREE_THRESHOLD:.0f}%)')
flag_handle = plt.scatter([],[],marker='D',color='grey',
                           edgecolors='k',s=50,label='Flagged (\u25c6)')
ok_handle   = plt.scatter([],[],marker='o',color='grey',
                           edgecolors='k',s=50,label='Not flagged (\u25cb)')
ax.legend(handles=class_handles+[thr_handle,flag_handle,ok_handle],
          fontsize=7, loc='upper right')

# ── Right: flagging rate per regime per class ──────────────────────────────
ax2 = axes[1]
regimes  = ['A (<15 m)', 'B (15-30 m)', 'C (>30 m)']
classes  = sorted(seen_classes)
n_reg    = len(regimes)
n_cls    = len(classes)
bar_w    = 0.8 / max(n_cls, 1)

for ci, cat in enumerate(classes):
    rates = []
    for reg in regimes:
        subset = [r for r in records
                  if r['category'] == cat and regime(r['range']) == reg]
        rates.append(100 * sum(r['flagged'] for r in subset) / len(subset)
                     if subset else 0.0)
    xs = np.arange(n_reg) + ci * bar_w - (n_cls - 1) * bar_w / 2
    ax2.bar(xs, rates, width=bar_w * 0.9,
            color=CLASS_COLORS.get(cat, 'grey'),
            edgecolor='k', linewidth=0.4, label=cat)

ax2.set_xticks(np.arange(n_reg))
ax2.set_xticklabels(regimes)
ax2.set_ylabel('Predictions flagged as overshoot [%]')
ax2.set_ylim(0, 105)
ax2.set_title(f'Flagging rate per regime (threshold={100*FREE_THRESHOLD:.0f}%)')
ax2.legend(fontsize=8)
ax2.axhline(50, color='black', lw=0.8, ls=':', alpha=0.5)

plt.tight_layout()
plt.show()

# ── Text summary ──────────────────────────────────────────────────────────
print('Flagging rate by regime:')
for reg in regimes:
    subset = [r for r in records if regime(r['range']) == reg]
    if not subset: continue
    n_flag = sum(r['flagged'] for r in subset)
    print(f'  {reg}: {n_flag}/{len(subset)} = {100*n_flag/len(subset):.0f}%  '
          f'(median free%: {100*np.median([r["frac_free"] for r in subset]):.1f}%)')


## 14. Phase 0.4 — Aggregate Experiment A (all annotated frames)

Run the batch script first from a terminal — **as two separate invocations, never `--dataset both`**
(loading NuScenes for ECP alone already costs ~7.4GB; running both datasets in one process risks OOM —
see the notebook's own §12.3 memory-safety note and `phase0_results.md` §3.3):
```bash
conda activate autolabeling
cd Contribution/phase0/scripts
python experiment_a_batch.py --dataset ecp
python experiment_a_batch.py --dataset nuscenes_mini
```
Then run the cell below to load results and plot the aggregate distribution.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

RESULTS_DIR = Path('../results')
FREE_THR    = 0.10
CLASS_COLORS = {
    'car'       : '#2196F3',
    'truck'     : '#9C27B0',
    'trailer'   : '#795548',
    'bus'       : '#FF5722',
    'pedestrian': '#4CAF50',
    'bicycle'   : '#FF9800',
    'motorcycle': '#F44336',
}

dfs = []
for ds in ['ecp', 'nuscenes_mini']:
    p = RESULTS_DIR / f'experiment_a_{ds}.csv'
    if p.exists():
        dfs.append(pd.read_csv(p))
        print(f'{ds}: {len(dfs[-1])} rows loaded')
    else:
        print(f'WARNING: {p} not found — run the batch script first')

if not dfs:
    raise RuntimeError('No CSV files found. Run experiment_a_batch.py first.')

df = pd.concat(dfs, ignore_index=True)
print(f'Total: {len(df)} predictions across '
      f'{df["sample_token"].nunique()} frames')
print('\nFlagging rate by dataset and regime:')
for ds in df['dataset'].unique():
    sub = df[df['dataset'] == ds]
    for reg in ['A', 'B', 'C']:
        s = sub[sub['regime'] == reg]
        if len(s) == 0: continue
        rate = s['flagged'].mean() * 100
        med  = s['frac_free'].median() * 100
        print(f'  {ds} Regime {reg}: {s["flagged"].sum()}/{len(s)} '
              f'= {rate:.0f}%  (median free%: {med:.1f}%)')

# ── Figure: 2 rows (one per dataset) × 2 cols (scatter + bar) ───────────
datasets  = df['dataset'].unique()
regimes   = ['A', 'B', 'C']
regime_labels = {'A': 'A (<15 m)', 'B': 'B (15-30 m)', 'C': 'C (>30 m)'}
fig, axes = plt.subplots(len(datasets), 2,
                         figsize=(14, 5 * len(datasets)))
if len(datasets) == 1:
    axes = axes[np.newaxis, :]

for row_i, ds in enumerate(datasets):
    sub    = df[df['dataset'] == ds]
    classes = sorted(sub['category'].unique())

    # ── Scatter ──────────────────────────────────────────────────────
    ax = axes[row_i, 0]
    ax.axvspan(0,    15, alpha=0.07, color='red')
    ax.axvspan(15,   30, alpha=0.07, color='orange')
    ax.axvspan(30,   55, alpha=0.07, color='grey')
    ax.axhline(FREE_THR * 100, color='black', lw=1.2, ls='--')
    ax.text(7,  97, 'Regime A', ha='center', fontsize=8, color='darkred')
    ax.text(22, 97, 'Regime B', ha='center', fontsize=8, color='darkorange')
    ax.text(42, 97, 'Regime C', ha='center', fontsize=8, color='dimgrey')

    for cat in classes:
        cdf = sub[sub['category'] == cat]
        col = CLASS_COLORS.get(cat, 'grey')
        ax.scatter(cdf['range_m'], cdf['frac_free'] * 100,
                   color=col, s=6, alpha=0.5, label=cat)

    ax.set_xlim(0, 55); ax.set_ylim(-2, 102)
    ax.set_xlabel('Range to ego [m]')
    ax.set_ylabel('OBB free-space voxels [%]')
    ax.set_title(f'{ds} — scatter (n={len(sub)})')
    handles = [mpatches.Patch(color=CLASS_COLORS.get(c,'grey'), label=c)
               for c in classes]
    handles.append(plt.Line2D([0],[0],color='black',ls='--',
                              label=f'Threshold ({100*FREE_THR:.0f}%)'))
    ax.legend(handles=handles, fontsize=7, loc='upper right')

    # ── Bar: flagging rate per regime per class ───────────────────────
    ax2 = axes[row_i, 1]
    n_cls = len(classes)
    bar_w = 0.8 / max(n_cls, 1)
    for ci, cat in enumerate(classes):
        rates = []
        for reg in regimes:
            s = sub[(sub['category'] == cat) & (sub['regime'] == reg)]
            rates.append(100 * s['flagged'].mean() if len(s) else 0.0)
        xs = np.arange(len(regimes)) + ci*bar_w - (n_cls-1)*bar_w/2
        ax2.bar(xs, rates, width=bar_w*0.9,
                color=CLASS_COLORS.get(cat,'grey'),
                edgecolor='k', linewidth=0.4, label=cat)

    ax2.set_xticks(np.arange(len(regimes)))
    ax2.set_xticklabels([regime_labels[r] for r in regimes])
    ax2.set_ylabel('Predictions flagged [%]')
    ax2.set_ylim(0, 105)
    ax2.axhline(50, color='black', lw=0.8, ls=':', alpha=0.4)
    ax2.set_title(f'{ds} — flagging rate per regime')
    ax2.legend(fontsize=8)

fig.suptitle('Phase 0.4 — OBB free% across all annotated frames '
             f'(threshold={100*FREE_THR:.0f}%)', fontsize=13)
plt.tight_layout()
out_path = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / 'phase0_aggregate_scatter_bar.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()


## 15. Phase 0.5 — GT Baseline: sensor-geometry floor

Run the same OBB free-space query on **annotated GT boxes** for every frame.
GT boxes are correctly sized, so their OBB free% captures the irreducible
sensor-geometry floor — how much certified-free space a *well-fitted* box
accumulates purely from beam density and voxel resolution.

The **corrected overshoot** signal is then:
```
excess_free% = predicted_free% − median_GT_free%(range_bin, sensor)
```
This is sensor-agnostic: ECP (64-beam) and nuScenes (32-beam) become directly comparable.

**Prerequisite:** re-run `experiment_a_batch.py` to generate `experiment_a_gt_baseline_<dataset>.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RESULTS_DIR = Path('../results')
BIN_WIDTH   = 5.0
EXCESS_THR  = 0.05   # flag if excess > 5 pp above GT median

CLASS_COLORS = {
    'car': '#2196F3', 'truck': '#9C27B0', 'trailer': '#795548',
    'bus': '#FF5722', 'pedestrian': '#4CAF50', 'bicycle': '#FF9800',
    'motorcycle': '#F44336', 'construction_vehicle': '#607D8B',
}

# ── Category mapping: nuScenes GT hierarchical → prediction label ─────────
# ECP GT already uses prediction names (car, bicycle, pedestrian, …)
NUSCENES_GT_TO_PRED = {
    'vehicle.car':                          'car',
    'vehicle.truck':                        'truck',
    'vehicle.bus.bendy':                    'bus',
    'vehicle.bus.rigid':                    'bus',
    'vehicle.trailer':                      'trailer',
    'vehicle.motorcycle':                   'motorcycle',
    'vehicle.bicycle':                      'bicycle',
    'vehicle.construction':                 'construction_vehicle',
    'human.pedestrian.adult':               'pedestrian',
    'human.pedestrian.child':               'pedestrian',
    'human.pedestrian.construction_worker': 'pedestrian',
    'human.pedestrian.personal_mobility':   'pedestrian',
    'human.pedestrian.police_officer':      'pedestrian',
    # movable_object.* → not in prediction label space, dropped
}

# ── Load CSVs ─────────────────────────────────────────────────────────────
gt_dfs, pred_dfs = [], []
for ds in ['ecp', 'nuscenes_mini']:
    gp = RESULTS_DIR / f'experiment_a_gt_baseline_{ds}.csv'
    pp = RESULTS_DIR / f'experiment_a_{ds}.csv'
    if gp.exists() and pp.exists():
        gt_dfs.append(pd.read_csv(gp))
        pred_dfs.append(pd.read_csv(pp))
        print(f'{ds}: {len(gt_dfs[-1])} GT rows, {len(pred_dfs[-1])} pred rows')
    else:
        print(f'WARNING: {ds} GT baseline not found — re-run experiment_a_batch.py')

if not gt_dfs:
    raise RuntimeError('No GT baseline CSVs found.')

df_gt   = pd.concat(gt_dfs,   ignore_index=True)
df_pred = pd.concat(pred_dfs, ignore_index=True)

# ── Map GT categories to prediction label space ───────────────────────────
def map_gt_category(row):
    if row['dataset'] == 'ecp':
        return row['category']           # already matches
    return NUSCENES_GT_TO_PRED.get(row['category'], None)

df_gt['category_pred'] = df_gt.apply(map_gt_category, axis=1)
df_gt = df_gt[df_gt['category_pred'].notna()].copy()

df_gt['range_bin']   = (df_gt['range_m']   // BIN_WIDTH * BIN_WIDTH).astype(float)
df_pred['range_bin'] = (df_pred['range_m'] // BIN_WIDTH * BIN_WIDTH).astype(float)

# ── Category-aware GT baseline (dataset × category × range_bin) ───────────
gt_baseline = (
    df_gt.groupby(['dataset', 'category_pred', 'range_bin'])['frac_free']
    .median()
    .rename('gt_median_free')
    .reset_index()
    .rename(columns={'category_pred': 'category'})
)

df_pred = df_pred.merge(gt_baseline, on=['dataset', 'category', 'range_bin'], how='left')
df_pred['excess_free']    = df_pred['frac_free'] - df_pred['gt_median_free'].fillna(0)
df_pred['excess_flagged'] = df_pred['excess_free'] > EXCESS_THR

# ── Per-category stats ────────────────────────────────────────────────────
print(f"\n{'dataset':<16} {'regime':<6} {'category':<22} {'n':>4}  {'P50':>6}  {'P90':>6}  {'>20pp':>6}")
print('-' * 72)
for ds in ['ecp', 'nuscenes_mini']:
    sub = df_pred[df_pred['dataset'] == ds]
    for reg in ['A', 'B', 'C']:
        s = sub[sub['regime'] == reg]
        if not len(s): continue
        for cat in sorted(s['category'].unique()):
            sc = s[s['category'] == cat]
            p50  = sc['excess_free'].median() * 100
            p90  = sc['excess_free'].quantile(0.90) * 100
            gt20 = (sc['excess_free'] > 0.20).mean() * 100
            print(f'{ds:<16} {reg:<6} {cat:<22} {len(sc):>4}  {p50:>+6.1f}  {p90:>+6.1f}  {gt20:>5.0f}%')

print('\nRegime-level summary (category-aware):')
for ds in ['ecp', 'nuscenes_mini']:
    sub = df_pred[df_pred['dataset'] == ds]
    for reg in ['A', 'B', 'C']:
        s = sub[sub['regime'] == reg]
        if not len(s): continue
        raw  = s['flagged'].mean() * 100
        corr = s['excess_flagged'].mean() * 100
        p50  = s['excess_free'].median() * 100
        p90  = s['excess_free'].quantile(0.90) * 100
        print(f'  {ds} {reg}: raw={raw:.0f}%  corrected={corr:.0f}%  P50={p50:>+.1f}pp  P90={p90:>+.1f}pp')

# ── Figure 1: predicted vs GT baseline median + P50→P90 excess band ───────
DS_LABELS = {'ecp': 'ECP (64-beam)', 'nuscenes_mini': 'nuScenes mini (32-beam)'}
COLORS = {'ecp': ('#1565C0', '#90CAF9'), 'nuscenes_mini': ('#B71C1C', '#EF9A9A')}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for ds in ['ecp', 'nuscenes_mini']:
    col_pred, col_gt = COLORS[ds]
    pred_med = df_pred[df_pred['dataset']==ds].groupby('range_bin')['frac_free'].median() * 100
    gt_med   = df_gt[df_gt['dataset']==ds].groupby('range_bin')['frac_free'].median() * 100
    ax.plot(pred_med.index + BIN_WIDTH/2, pred_med.values,
            color=col_pred, lw=2, marker='o', ms=4, label=f'{DS_LABELS[ds]} — predicted')
    ax.plot(gt_med.index + BIN_WIDTH/2, gt_med.values,
            color=col_gt, lw=2, marker='s', ms=4, ls='--', label=f'{DS_LABELS[ds]} — GT baseline')
ax.axvline(15, color='grey', lw=0.8, ls=':')
ax.axvline(30, color='grey', lw=0.8, ls=':')
ax.text(7.5,  96, 'A', ha='center', color='grey', fontsize=9)
ax.text(22.5, 96, 'B', ha='center', color='grey', fontsize=9)
ax.text(42.5, 96, 'C', ha='center', color='grey', fontsize=9)
ax.set_xlabel('Range to ego [m]')
ax.set_ylabel('Median OBB free% [%]')
ax.set_title('Predicted vs GT baseline: median free% by range')
ax.legend(fontsize=8); ax.set_xlim(0, 55); ax.set_ylim(-2, 100)

ax2 = axes[1]
for ds in ['ecp', 'nuscenes_mini']:
    col_pred, _ = COLORS[ds]
    grp = df_pred[df_pred['dataset']==ds].groupby('range_bin')['excess_free']
    p50 = grp.median() * 100
    p90 = grp.quantile(0.90) * 100
    x   = p50.index + BIN_WIDTH/2
    ax2.plot(x, p50.values, color=col_pred, lw=2, marker='o', ms=4, label=f'{DS_LABELS[ds]} — P50')
    ax2.fill_between(x, p50.values, p90.values, color=col_pred, alpha=0.18,
                     label=f'{DS_LABELS[ds]} — P50→P90 band')
ax2.axhline(EXCESS_THR * 100, color='black', lw=1.2, ls='--',
            label=f'Flag margin ({int(EXCESS_THR*100)} pp)')
ax2.axhline(0, color='black', lw=0.6, alpha=0.3)
ax2.axvline(15, color='grey', lw=0.8, ls=':')
ax2.axvline(30, color='grey', lw=0.8, ls=':')
ax2.set_xlabel('Range to ego [m]')
ax2.set_ylabel('Excess free% above GT baseline [pp]')
ax2.set_title('Corrected overshoot: P50 ± P90 band (category-aware)')
ax2.legend(fontsize=8); ax2.set_xlim(0, 55)

fig.suptitle('Phase 0.5 — Category-aware GT-corrected free-space overshoot', fontsize=13)
plt.tight_layout()
_out1 = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / 'phase0_gt_corrected_excess.png'
plt.savefig(_out1, dpi=150, bbox_inches='tight')
print(f'Saved: {_out1}')
plt.tight_layout()
plt.show()

# ── Figure 2: P90 excess by category and regime ───────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
regimes = ['A', 'B', 'C']
regime_labels = {'A': 'A (<15m)', 'B': 'B (15-30m)', 'C': 'C (>30m)'}

for col_i, ds in enumerate(['ecp', 'nuscenes_mini']):
    ax = axes2[col_i]
    sub    = df_pred[df_pred['dataset'] == ds]
    classes = sorted(sub['category'].unique())
    n_cls   = len(classes)
    bar_w   = 0.8 / max(n_cls, 1)

    for ci, cat in enumerate(classes):
        p90s = []
        for reg in regimes:
            s = sub[(sub['category']==cat) & (sub['regime']==reg)]
            p90s.append(s['excess_free'].quantile(0.90)*100 if len(s) >= 5 else 0.0)
        xs = np.arange(len(regimes)) + ci*bar_w - (n_cls-1)*bar_w/2
        ax.bar(xs, p90s, width=bar_w*0.9,
               color=CLASS_COLORS.get(cat, 'grey'),
               edgecolor='k', linewidth=0.4, label=cat)

    ax.axhline(20, color='black', lw=0.8, ls=':', alpha=0.5, label='>20pp = severe')
    ax.set_xticks(np.arange(len(regimes)))
    ax.set_xticklabels([regime_labels[r] for r in regimes])
    ax.set_ylabel('P90 excess free% above GT baseline [pp]')
    ax.set_ylim(0, 65)
    ax.set_title(f'{ds} — P90 corrected excess by category & regime')
    ax.legend(fontsize=7)

fig2.suptitle('Phase 0.5 — P90 corrected excess per class (category-aware GT baseline)', fontsize=13)
plt.tight_layout()
_out2 = DEV_ROOT / 'contribution_ideas' / 'phase0' / 'results' / 'phase0_gt_p90_by_category.png'
plt.savefig(_out2, dpi=150, bbox_inches='tight')
print(f'Saved: {_out2}')
plt.tight_layout()
plt.show()

# ── Save corrected CSV ────────────────────────────────────────────────────
out = RESULTS_DIR / 'experiment_a_corrected.csv'
df_pred[['dataset','sample_token','category','score','range_m','regime',
          'frac_free','gt_median_free','excess_free','excess_flagged']].to_csv(out, index=False)
print(f'\nCorrected results saved → {out}')
